# 03 — Bộ câu hỏi "trước/sau" có nhãn kiểu thay đổi (dữ liệu cho Trụ 2)

**Đề tài:** Trôi phiên bản tri thức trong trợ lý CTI dùng LLM/RAG

| Notebook | Việc | Đầu ra |
|---|---|---|
| `01_thay_doi_phien_ban_ATTACK` | Ground truth các thay đổi giữa các phiên bản ATT&CK | `drift/` |
| `02_benchmark_theo_phien_ban` | Benchmark hỏi–đáp cho **một** phiên bản (đã chạy cho v19.2 và v18.1) | `benchmark/attack-<phiên bản>/` |
| **`03_cap_cau_hoi_truoc_sau` (notebook này)** | Ghép **cùng một câu hỏi** ở hai phiên bản, gắn nhãn kiểu thay đổi | `benchmark/drift-18.1-19.2/` |
| `04` (tiếp theo) | Pilot: LLM không RAG / RAG một phiên bản / RAG trộn phiên bản | tỉ lệ trả lời lỗi thời theo kiểu thay đổi |
| `05` | Phương pháp nhận biết phiên bản (Trụ 3) | so sánh với các baseline |

## Notebook này làm gì?

Trụ 2 đặt câu hỏi: **khi tri thức đổi giữa hai phiên bản, LLM/RAG trả lời theo phiên bản nào, và loại thay đổi nào gây lỗi nhiều nhất?** Để đo được, mỗi mục dữ liệu phải có:

1. **Một câu hỏi cố định** (ví dụ: *"T1222 thuộc tactic nào?"*).
2. **Đáp án theo phiên bản cũ** (v18.1: *Defense Evasion – TA0005*) và **đáp án theo phiên bản mới** (v19.2: *Defense Impairment – TA0112*).
3. **Nhãn kiểu thay đổi** lấy từ notebook 01 (ở đây: `TACTIC_CHANGED`).
4. **Bằng chứng** trong kho RAG của *từng* phiên bản.

## Vì sao không ghép trực tiếp hai file `questions.jsonl` của notebook 02?

Notebook 02 **lấy mẫu** tối đa 400 câu mỗi mẫu câu hỏi, nên phần lớn kỹ thuật bị thay đổi (T1562, T1548.x…) **không nằm trong mẫu**. Ghép hai mẫu ngẫu nhiên sẽ bỏ sót đúng những mục ta cần đo. Vì vậy notebook này sinh lại câu hỏi trên **toàn bộ quần thể** kỹ thuật của cả hai phiên bản, dùng **cùng mẫu câu, cùng seed, cùng cách chia dev/test** với notebook 02. Sau đó mới lấy mẫu cho pilot, theo tầng (*stratified*) để mỗi kiểu thay đổi đều có mặt.

## Các lớp cặp câu hỏi (`pair_class`)

| Lớp | Ý nghĩa | Ví dụ v18.1 → v19.2 |
|---|---|---|
| `STABLE` | Mã còn hiệu lực, đáp án và nhãn hiển thị không đổi. Đây là **nhóm đối chứng**. | T1001 → Command and Control |
| `ANSWER_CHANGED` | Mã còn hiệu lực nhưng **tập đáp án đổi** | T1222: TA0005 → TA0112 |
| `LABEL_CHANGED` | Đáp án giữ **nguyên mã** nhưng **tên** của mã đổi. Đây là trường hợp *cùng mã, đổi nghĩa*. | T1027 vẫn thuộc TA0005, nhưng TA0005 nay tên là *Stealth* |
| `ANCHOR_RENAMED` | Thực thể được hỏi bị đổi tên, đáp án không đổi | T1222.001 đổi tên thành *Windows Permissions* |
| `REVOKED_ANCHOR` | Mã trong câu hỏi **bị thu hồi**; đáp án hiện hành lấy theo mã thay thế | T1562.004 → T1686 |
| `NEW_ONLY` | Thực thể **chỉ có ở phiên bản mới**; mô hình huấn luyện trước v19 không thể biết | T1685 Disable or Modify Tools |

## Bốn mẫu câu hỏi

| Mẫu | Hỏi gì | Kiểu đáp án |
|---|---|---|
| `D1_tech_tactic` | Kỹ thuật thuộc tactic nào | tập mã TA |
| `D2_subtech_parent` | Sub-technique (hỏi **bằng tên**) có kỹ thuật cha nào | tập mã T |
| `D3_tech_mitigation` | Kỹ thuật có những biện pháp giảm thiểu nào | tập mã M |
| `D4_id_currency` | Mã kỹ thuật còn hiệu lực không, nếu không thì mã thay thế là gì | tập mã T |

`D4` là mẫu mới, không có trong notebook 02. Nó đo trực tiếp hiện tượng **"thẻ lỗi thời"** (*stale tag*) mà Trụ 1 đo ngoài thực tế: nhà phân tích cầm một mã cũ trong báo cáo hay luật phát hiện và hỏi trợ lý.

## Ba cách hỏi (`framing`) cho mỗi mục

| Framing | Câu hỏi | Đáp án đúng | Lỗi trôi phiên bản |
|---|---|---|---|
| `unversioned` | Không nhắc phiên bản (cách hỏi tự nhiên nhất) | theo **bản mới** | trả lời theo bản cũ = **STALE** |
| `current` | Nói rõ "bản hiện hành v19.2" | theo **bản mới** | trả lời theo bản cũ = **STALE** |
| `historical` | Nói rõ "trong v18.1" | theo **bản cũ** | trả lời theo bản mới = **ANACHRONISM** |

So sánh `unversioned` với `current` cho biết **nói rõ phiên bản có giúp mô hình không**. Framing `historical` kiểm tra chiều ngược lại: hệ thống có trả lời được câu hỏi về quá khứ không, ví dụ khi đọc lại một báo cáo sự cố cũ.

## Quy ước quan trọng (phải ghi trong bài báo)

- Câu hỏi luôn nêu thực thể **như nó xuất hiện ở phiên bản cũ** (mã cũ, tên cũ). Kịch bản: nhà phân tích cầm một tài liệu thời v18.1 và hỏi trợ lý. Riêng `NEW_ONLY` dùng mã và tên mới.
- Đáp án đúng **không do LLM viết**. Nó được suy ra tất định từ STIX của từng phiên bản và theo chuỗi `revoked-by`, tức là **đúng ánh xạ chính thức của MITRE**. Ánh xạ này được **đối chiếu với bảng thu hồi trong release notes v19** (mục 4). Với v18.1 → v19.2, cả 17/17 mã khớp, và không có trường hợp một mã cũ được thay bằng nhiều mã mới.
- Chấm điểm bằng **mã chuẩn** (TA…, T…, M…). Riêng tactic chấp nhận cả tên, theo **cả tên cũ lẫn tên mới**, và ghi nhận riêng việc dùng **nhãn lỗi thời** (ví dụ nói "Defense Evasion" ở câu hỏi về bản hiện hành).

### Nhật ký phiên bản

| Phiên bản | Ngày | Thay đổi |
|---|---|---|
| 1.0 | 12/10/2026 | Bản đầu tiên |
| 1.2 | 12/10/2026 | **Đính chính:** release notes v19 **không** tách T1562 thành 5 kỹ thuật; T1562 được thay bằng **một** mã T1685, đúng như STIX. Bỏ yêu cầu lập `splits_manual.csv` cho v18.1→v19.2 (file chỉ còn là cơ chế tuỳ chọn cho các bản sau). Thêm phép đối chiếu tự động 17 mã thu hồi với bảng release notes. Dữ liệu đầu ra không đổi. |
| 1.1 | 12/10/2026 | Sửa theo kiểm tra độc lập. **Chia dev/test theo dòng dõi** (mã cũ + mã kế nhiệm cùng một tập), chống rò rỉ. D2 giữ các kỹ thuật cấp cha trở thành sub-technique (T1656→T1684.001, T1672→T1684.002) và yêu cầu trả lời bằng mã T. Nhãn thay đổi lọc theo mẫu câu hỏi; thêm `answer_changed`, `drift_possible`. Module chấm điểm: không phân biệt hoa/thường, đọc được danh sách nhiều dòng sau `ANSWER:`, che tên thực thể được hỏi, không suy tactic từ chữ thường trong câu văn. Đọc `splits_manual.csv` an toàn hơn; chạy được với khoảng phiên bản dài. Thêm kiểm tra số 7. |

## 1. Cấu hình và kiểm tra đầu vào

Chạy notebook **trong thư mục dự án** (ví dụ `~/p/cti-drift/`). Notebook này **không tải gì từ mạng**; nó chỉ dùng kết quả của notebook 01 và 02:

- `data/raw/attack_versions/enterprise-attack-<V>.json` (notebook 01 đã tải)
- `drift/change_events.csv` (notebook 01)
- `benchmark/attack-<V>/corpus.jsonl` của **cả hai** phiên bản (notebook 02, chạy hai lần)
- `drift/splits_manual.csv` (**tuỳ chọn**, chỉ cần khi release notes của một bản *sau này* nêu rõ một mã cũ được thay bằng **nhiều** mã mới; với v18.1 → v19.2 **không cần**)
- `data/raw/stix-capec.json` (tuỳ chọn, chỉ dùng ở phụ lục)

`SEED` và `DEV_PERCENT` **phải giống notebook 02** để một kỹ thuật luôn rơi vào cùng tập dev/test ở mọi notebook. Mục kiểm tra số 6 bên dưới sẽ xác nhận điều này.

In [1]:
from pathlib import Path
import json, re, hashlib, random, datetime, collections
import pandas as pd

V_OLD, V_NEW = "18.1", "19.2"      # phiên bản cũ / mới cần so sánh

ROOT    = Path.cwd()
RAW     = ROOT / "data" / "raw"
DRIFT   = ROOT / "drift"
OUT     = ROOT / "benchmark" / f"drift-{V_OLD}-{V_NEW}"
OUT.mkdir(parents=True, exist_ok=True)

SEED                  = 2026      # PHẢI giống notebook 02
DEV_PERCENT           = 20        # PHẢI giống notebook 02
MAX_MITIGATIONS       = 6         # D3: bỏ mục có quá nhiều mitigation (khó chấm, ít ý nghĩa)
CONTROLS_PER_TEMPLATE = 40        # pilot: số mục STABLE (đối chứng) mỗi mẫu câu hỏi
MAX_PER_CLASS         = 80        # pilot: trần mỗi lớp thay đổi mỗi mẫu câu hỏi
DRIFTSET_VERSION      = "1.2"

REQUIRED = {
    "attack_old":    RAW / "attack_versions" / f"enterprise-attack-{V_OLD}.json",
    "attack_new":    RAW / "attack_versions" / f"enterprise-attack-{V_NEW}.json",
    "change_events": DRIFT / "change_events.csv",
    "corpus_old":    ROOT / "benchmark" / f"attack-{V_OLD}" / "corpus.jsonl",
    "corpus_new":    ROOT / "benchmark" / f"attack-{V_NEW}" / "corpus.jsonl",
}
OPTIONAL = {"splits_manual": DRIFT / "splits_manual.csv", "capec": RAW / "stix-capec.json"}

missing = [f"{k}: {p}" for k, p in REQUIRED.items() if not p.exists()]
assert not missing, "Thiếu đầu vào — hãy chạy notebook 01 và 02 (cả hai phiên bản) trước:\n  " + "\n  ".join(missing)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

MANIFEST = {"driftset_version": DRIFTSET_VERSION, "v_old": V_OLD, "v_new": V_NEW,
            "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "config": dict(SEED=SEED, DEV_PERCENT=DEV_PERCENT, MAX_MITIGATIONS=MAX_MITIGATIONS,
                           CONTROLS_PER_TEMPLATE=CONTROLS_PER_TEMPLATE, MAX_PER_CLASS=MAX_PER_CLASS),
            "inputs": {}}
for k, p in {**REQUIRED, **OPTIONAL}.items():
    if p.exists():
        MANIFEST["inputs"][k] = {"path": str(p.relative_to(ROOT)), "sha256": sha256(p)}
        print(f"{k:14s} {MANIFEST['inputs'][k]['sha256'][:12]}…  {p.relative_to(ROOT)}")
    else:
        print(f"{k:14s} (không có)      {p.relative_to(ROOT)}")

attack_old     f857d8f78f2f…  data/raw/attack_versions/enterprise-attack-18.1.json
attack_new     dc1639caa550…  data/raw/attack_versions/enterprise-attack-19.2.json
change_events  ab7f97888bf7…  drift/change_events.csv
corpus_old     d435b77a7e8f…  benchmark/attack-18.1/corpus.jsonl
corpus_new     0198ec96b46b…  benchmark/attack-19.2/corpus.jsonl
splits_manual  (không có)      drift/splits_manual.csv
capec          ee6244f48259…  data/raw/stix-capec.json


## 2. Đọc hai phiên bản ATT&CK

Hàm `load_snapshot` giống hệt notebook 01. Ta thêm bảng mitigation theo kỹ thuật để dùng cho mẫu D3.

In [2]:
def ext_id(o):
    for r in o.get("external_references", []):
        if r.get("source_name") == "mitre-attack" and r.get("external_id"):
            return r["external_id"]

def status(o):
    if o.get("revoked"):
        return "revoked"
    if o.get("x_mitre_deprecated"):
        return "deprecated"
    return "active"

def load_snapshot(v):
    objs = json.loads((RAW / "attack_versions" / f"enterprise-attack-{v}.json").read_text())["objects"]
    s2i, tactics, tech, mit = {}, {}, {}, {}
    for o in objs:
        i = ext_id(o)
        if i:
            s2i[o["id"]] = i
        if o["type"] == "x-mitre-tactic" and status(o) == "active":
            tactics[o["x_mitre_shortname"]] = (i, o["name"])
    ta_by_id = {tid: (sn, name) for sn, (tid, name) in tactics.items()}
    for o in objs:
        i = ext_id(o)
        if not i:
            continue
        if o["type"] == "attack-pattern":
            tac = sorted({tactics[k["phase_name"]][0] for k in o.get("kill_chain_phases", [])
                          if k.get("kill_chain_name") == "mitre-attack" and k["phase_name"] in tactics})
            tech[i] = dict(stix=o["id"], name=o["name"], status=status(o), tactics=tuple(tac),
                           is_sub=bool(o.get("x_mitre_is_subtechnique")), parent=None,
                           version=o.get("x_mitre_version", "1.0"))
        elif o["type"] == "course-of-action":
            mit[i] = dict(name=o["name"], status=status(o))
    revoked_by, mitigates = {}, set()
    for o in objs:
        if o["type"] != "relationship":
            continue
        src, dst, rt = s2i.get(o["source_ref"]), s2i.get(o["target_ref"]), o["relationship_type"]
        if not src or not dst:
            continue
        if rt == "revoked-by":
            revoked_by[src] = dst
        elif status(o) != "active":
            continue
        elif rt == "subtechnique-of" and src in tech:
            tech[src]["parent"] = dst
        elif rt == "mitigates" and src in mit and dst in tech:
            if mit[src]["status"] == "active" and tech[dst]["status"] == "active":
                mitigates.add((src, dst))
    mits_of = collections.defaultdict(set)
    for m, t in mitigates:
        mits_of[t].add(m)
    return dict(version=v, tactics=ta_by_id, tech=tech, mit=mit, revoked_by=revoked_by,
                mitigates=mitigates, mits_of=mits_of)

def resolve(snapshot, i, max_hops=10):
    # Theo chuỗi revoked-by trong CÙNG một phiên bản tới khi gặp đối tượng còn hiệu lực
    path = [i]
    while snapshot["tech"].get(i, {}).get("status") == "revoked" and i in snapshot["revoked_by"] and len(path) <= max_hops:
        i = snapshot["revoked_by"][i]
        path.append(i)
    return i, path

OLD, NEW = load_snapshot(V_OLD), load_snapshot(V_NEW)
act = lambda S: {i for i, t in S["tech"].items() if t["status"] == "active"}
ACT_OLD, ACT_NEW = act(OLD), act(NEW)
for S, A in ((OLD, ACT_OLD), (NEW, ACT_NEW)):
    print(f"v{S['version']}: tactic={len(S['tactics'])}, kỹ thuật active={len(A)} "
          f"(con={sum(S['tech'][i]['is_sub'] for i in A)}), quan hệ mitigates={len(S['mitigates'])}")

v18.1: tactic=14, kỹ thuật active=691 (con=475), quan hệ mitigates=1445
v19.2: tactic=15, kỹ thuật active=697 (con=475), quan hệ mitigates=1448


## 3. Bảng tách thủ công `splits_manual.csv` (tuỳ chọn — v18.1 → v19.2 KHÔNG cần)

Trong STIX, mỗi mã bị thu hồi chỉ trỏ tới **một** mã thay thế (`revoked-by`). Nếu một bản ATT&CK nào đó **tách** một kỹ thuật cũ thành nhiều kỹ thuật mới (1→N), STIX sẽ không diễn tả đủ. Khi đó ta cần một bảng bổ sung lấy từ release notes.

**Với v18.1 → v19.2, bảng này không cần.** Bảng thu hồi trong release notes v19 ghi mỗi mã cũ ứng với **đúng một** mã mới, và khớp 17/17 với STIX (mục 4 kiểm tra tự động). Có những trường hợp **nhiều mã cũ gộp vào một mã mới** (N→1): T1562, T1562.001, T1562.006 → T1685. STIX diễn tả được trường hợp này, nên không cần bảng.

Cơ chế này được giữ lại cho các bản ATT&CK sau. Định dạng file (cột `new_ids` cách nhau bằng dấu `;`):

```csv
old_id,new_ids,note
TXXXX,TYYYY;TZZZZ,<trích dẫn release notes có nói rõ việc tách>
```

Notebook kiểm tra: mã cũ phải **bị thu hồi** ở bản mới, mọi mã mới phải **còn hiệu lực**, và đích `revoked-by` của STIX phải nằm trong danh sách. Chỉ thêm một dòng khi **release notes nói rõ** việc tách. Không suy diễn, kể cả khi các sub-technique của mã cũ được chuyển sang nhiều kỹ thuật mới khác nhau.

In [3]:
SPLITS = {}
p = OPTIONAL["splits_manual"]
if p.exists() and p.stat().st_size > 0:
    # index_col=False: dấu phẩy thừa trong cột note không làm lệch cột; utf-8-sig: chấp nhận file có BOM (Excel)
    sm = pd.read_csv(p, dtype=str, index_col=False, encoding="utf-8-sig", skipinitialspace=True).fillna("")
    sm.columns = [c.strip() for c in sm.columns]
    assert {"old_id", "new_ids"} <= set(sm.columns), (f"splits_manual.csv cần cột old_id, new_ids (phân cách bằng dấu PHẨY); "
                                                      f"đang đọc được {list(sm.columns)}")
    sm = sm[sm.old_id.str.strip() != ""]                   # bỏ dòng trống (Excel hay để lại ",,")
    dup = sm.old_id.str.strip()[sm.old_id.str.strip().duplicated()].tolist()
    assert not dup, f"splits_manual.csv có old_id lặp lại: {dup}"
    for _, r in sm.iterrows():
        old = r["old_id"].strip().upper()
        news = sorted({x.strip().upper() for x in r["new_ids"].split(";") if x.strip()})
        assert NEW["tech"].get(old, {}).get("status") == "revoked", f"{old}: không bị thu hồi ở v{V_NEW}"
        bad = [n for n in news if n not in ACT_NEW]
        assert not bad, f"{old}: các mã mới không còn hiệu lực ở v{V_NEW}: {bad}"
        tgt, _ = resolve(NEW, old)
        assert tgt in news, f"{old}: đích revoked-by của STIX ({tgt}) không có trong danh sách {news}"
        if old not in ACT_OLD:
            print(f"⚠ {old} không còn hiệu lực ở v{V_OLD} — dòng này không ảnh hưởng tới cặp v{V_OLD}→v{V_NEW}")
        SPLITS[old] = news
    print(f"Đã nạp {len(SPLITS)} trường hợp tách 1→N:", SPLITS)
else:
    print("Không có drift/splits_manual.csv → dùng đúng ánh xạ revoked-by chính thức của MITRE (đúng với v18.1 → v19.2).")
MANIFEST["splits_manual_n"] = len(SPLITS)

Không có drift/splits_manual.csv → dùng đúng ánh xạ revoked-by chính thức của MITRE (đúng với v18.1 → v19.2).


## 4. Mỗi mã cũ ứng với mã nào ở bản mới?

Với mỗi kỹ thuật còn hiệu lực ở v18.1, ta xác định **mã kế nhiệm** ở v19.2:

- vẫn còn hiệu lực → chính nó (`same`)
- bị thu hồi và có trong `splits_manual.csv` → danh sách của em (`splits_manual`)
- bị thu hồi → theo chuỗi `revoked-by` (`revoked-by`)
- bị ngừng dùng (*deprecated*), không có mã thay thế → không có kế nhiệm

Sau đó **đối chiếu với notebook 01**. Tập mã bị thu hồi tính ở đây phải trùng với các sự kiện `REVOKED→*` mà notebook 01 ghi nhận trong khoảng v18.1 → v19.2.

In [4]:
def successors(old_id):
    st = NEW["tech"].get(old_id, {}).get("status")
    if st == "active":
        return [old_id], "same"
    if st == "revoked":
        tgt, path = resolve(NEW, old_id)
        for x in path:                      # mã (hoặc một mắt xích trung gian) có trong bảng tách thủ công
            if x in SPLITS:
                return SPLITS[x], "splits_manual"
        return ([tgt] if tgt in ACT_NEW else []), "revoked-by"
    return [], st or "missing"

SUCC = {i: successors(i) for i in sorted(ACT_OLD)}
src_count = collections.Counter(s for _, s in SUCC.values())
print("Nguồn mã kế nhiệm:", dict(src_count))

# Sự kiện của notebook 01 nằm trong khoảng (V_OLD, V_NEW]
def vkey(v):
    return tuple(int(x) for x in str(v).split("."))
EV = pd.read_csv(REQUIRED["change_events"], dtype=str).fillna("")
EV = EV[(EV.v_from.map(vkey) >= vkey(V_OLD)) & (EV.v_to.map(vkey) <= vkey(V_NEW))]
print(f"Sự kiện notebook 01 trong khoảng v{V_OLD} → v{V_NEW}:", EV.type.value_counts().to_dict())

revoked_here = {i for i, (_, s) in SUCC.items() if s in ("splits_manual", "revoked-by")}
# Chỉ so với các mã còn hiệu lực ở V_OLD: khi khoảng so sánh dài, có mã được thêm rồi bị thu hồi ngay trong khoảng
revoked_nb01 = set(EV[EV.type.str.startswith("REVOKED")].id_old) & ACT_OLD
assert revoked_here == revoked_nb01, f"Lệch với notebook 01: {sorted(revoked_here ^ revoked_nb01)}"
print(f"✓ {len(revoked_here)} mã bị thu hồi — khớp notebook 01")

# Đối chiếu với BẢNG THU HỒI trong release notes chính thức ATT&CK v19 (Enterprise, mục "Revocations"):
# https://attack.mitre.org/resources/updates/updates-april-2026/  — chép tay, mỗi mã cũ -> mã thay thế
RELEASE_NOTES_V19 = {
    "T1070.002": "T1685.006", "T1070.001": "T1685.005", "T1562.002": "T1685.001", "T1562.007": "T1686.001",
    "T1562.008": "T1685.002", "T1562.012": "T1685.004", "T1562.013": "T1686.002", "T1562.004": "T1686",
    "T1562.001": "T1685",     "T1562.010": "T1689",     "T1672": "T1684.002",     "T1562.003": "T1690",
    "T1562": "T1685",         "T1656": "T1684.001",     "T1562.006": "T1685",     "T1562.009": "T1688",
    "T1562.011": "T1685.003",
}
if V_OLD == "18.1" and V_NEW.startswith("19."):
    stix_map = {i: resolve(NEW, i)[0] for i in revoked_here}
    diff_rn = {i: (stix_map.get(i), RELEASE_NOTES_V19.get(i)) for i in set(stix_map) | set(RELEASE_NOTES_V19)
               if stix_map.get(i) != RELEASE_NOTES_V19.get(i)}
    assert not diff_rn, f"STIX khác release notes: {diff_rn}"
    print(f"✓ Ánh xạ revoked-by của STIX khớp release notes v19: {len(RELEASE_NOTES_V19)}/{len(RELEASE_NOTES_V19)} mã, "
          "không có trường hợp 1→N")

# Tra nhãn kiểu thay đổi của notebook 01 cho một mã
EV_BY_ID = collections.defaultdict(set)
for _, r in EV.iterrows():
    for i in set(r.id_old.split(",")) | {r.id_new}:
        if i:
            EV_BY_ID[i].add(r.type)
REDEFINED_TACTICS = set(EV[EV.type == "TACTIC_REDEFINED"].id_old) | {
    t for t in set(OLD["tactics"]) & set(NEW["tactics"]) if OLD["tactics"][t][1] != NEW["tactics"][t][1]}
print("Tactic đổi nghĩa / đổi tên (cùng mã):", {t: (OLD['tactics'][t][1], NEW['tactics'][t][1]) for t in sorted(REDEFINED_TACTICS)})

Nguồn mã kế nhiệm: {'same': 674, 'revoked-by': 17}
Sự kiện notebook 01 trong khoảng v18.1 → v19.2: {'MAJOR_REVISION': 199, 'TACTIC_CHANGED': 72, 'ADDED': 23, 'REVOKED→NEW': 17, 'RENAMED': 4, 'MERGE': 1, 'REFINED': 1, 'TACTIC_REDEFINED': 1, 'TACTIC_ADDED': 1}
✓ 17 mã bị thu hồi — khớp notebook 01
✓ Ánh xạ revoked-by của STIX khớp release notes v19: 17/17 mã, không có trường hợp 1→N
Tactic đổi nghĩa / đổi tên (cùng mã): {'TA0005': ('Defense Evasion', 'Stealth')}


## 5. Kho RAG của hai phiên bản và cách tìm bằng chứng

Dùng lại `corpus.jsonl` của notebook 02. Mỗi đoạn có mã dạng `T1001#0` (đoạn đầu tiên của tài liệu T1001).

| Mẫu | Bằng chứng |
|---|---|
| D1 | đoạn `#0` của kỹ thuật (chứa dòng `Tactics: …`) |
| D2 | đoạn `#0` của sub-technique (chứa dòng `Sub-technique of: …`) |
| D3 | đoạn `#0` của kỹ thuật + các đoạn mitigation chứa dòng `Mitigates <mã kỹ thuật>` |
| D4 | đoạn `#0` của mã được hỏi, **nếu mã đó còn trong kho** |

**Phát hiện cần ghi lại:** kho của bản mới **không còn nhắc đến mã đã bị thu hồi**. Ô dưới đếm số lần mỗi mã bị thu hồi xuất hiện trong kho v19.2. Nếu bằng 0, một hệ RAG chỉ dùng bản mới **không thể tra ngược** từ mã cũ (ví dụ T1562.004) sang mã mới bằng tìm kiếm từ khoá. Đây là động cơ trực tiếp cho phương pháp ở Trụ 3.

In [5]:
def load_corpus(p):
    return {c["_id"]: c for c in map(json.loads, open(p, encoding="utf-8"))}

CORPUS = {V_OLD: load_corpus(REQUIRED["corpus_old"]), V_NEW: load_corpus(REQUIRED["corpus_new"])}
MIT_CHUNKS = {}
for v, C in CORPUS.items():
    idx = collections.defaultdict(list)
    for cid, c in C.items():
        if c["metadata"]["kind"] == "mitigation":
            m = c["metadata"]["doc_id"]
            for t in re.findall(r"Mitigates (T\d{4}(?:\.\d{3})?)\b", c["text"]):
                idx[(m, t)].append(cid)
    MIT_CHUNKS[v] = idx
    print(f"Kho v{v}: {len(C)} đoạn")

all_new_text = "\n".join(c["text"] for c in CORPUS[V_NEW].values())
mentions = {i: len(re.findall(re.escape(i) + r"(?![.\d])", all_new_text)) for i in sorted(revoked_here)}
print(f"Số lần các mã bị thu hồi xuất hiện trong kho v{V_NEW}:", sum(mentions.values()), mentions)
MANIFEST["revoked_ids_mentioned_in_new_corpus"] = sum(mentions.values())

def ev_tech(v, ids):
    return [f"{i}#0" for i in ids if f"{i}#0" in CORPUS[v]]

def ev_mit(v, tech_ids, mits):
    out = ev_tech(v, tech_ids)
    for t in tech_ids:
        for m in sorted(mits):
            out += MIT_CHUNKS[v].get((m, t), [])
    return list(dict.fromkeys(out))

Kho v18.1: 4707 đoạn
Kho v19.2: 4738 đoạn
Số lần các mã bị thu hồi xuất hiện trong kho v19.2: 0 {'T1070.001': 0, 'T1070.002': 0, 'T1562': 0, 'T1562.001': 0, 'T1562.002': 0, 'T1562.003': 0, 'T1562.004': 0, 'T1562.006': 0, 'T1562.007': 0, 'T1562.008': 0, 'T1562.009': 0, 'T1562.010': 0, 'T1562.011': 0, 'T1562.012': 0, 'T1562.013': 0, 'T1656': 0, 'T1672': 0}


## 6. Sinh các mục ghép cặp

Mỗi **mục** (item) là một câu hỏi kèm đáp án ở cả hai phiên bản. Các trường chính:

| Trường | Ý nghĩa |
|---|---|
| `item_id` | ví dụ `D1-T1222`: mẫu + mã thực thể (ổn định, đọc được) |
| `anchor_old`, `anchor_new` | mã được hỏi ở bản cũ; (các) mã tương ứng ở bản mới |
| `gold_old`, `gold_new` | đáp án theo v18.1 và theo v19.2 |
| `removed`, `added` | `gold_old − gold_new` (đáp án **lỗi thời**) và `gold_new − gold_old` (đáp án **mới**) |
| `pair_class`, `change_types` | lớp cặp (bảng ở đầu notebook) và nhãn kiểu thay đổi của notebook 01 |
| `questions` | ba cách hỏi: `unversioned`, `current`, `historical` |
| `evidence` | mã đoạn bằng chứng trong kho **từng** phiên bản |
| `old_id_in_new_corpus` | mã cũ có còn xuất hiện trong kho bản mới không |
| `answer_changed` | `True` nếu tập đáp án đổi giữa hai phiên bản, tức là **có thể** trả lời lỗi thời. Một số mục `REVOKED_ANCHOR` có mã đổi nhưng đáp án giữ nguyên (ví dụ mitigation của mã cũ và mã mới trùng nhau). Khi tính tỉ lệ STALE phải tách riêng các mục này để không làm loãng kết quả. |
| `change_types` | nhãn của notebook 01 **liên quan tới mẫu câu hỏi này** (ví dụ D2 không mang nhãn `TACTIC_CHANGED`); các sự kiện khác của thực thể nằm ở `other_events` |
| `lineage`, `split` | nhóm dòng dõi (mã cũ + mã kế nhiệm) và tập dev/test của nhóm |

**Chia dev/test theo dòng dõi.** Mã cũ và mã kế nhiệm mang cùng một sự kiện. Ví dụ T1562.002 và T1685.001 có cùng đáp án hiện hành. Nếu chia theo từng mã riêng, sự kiện đó có thể nằm ở cả dev lẫn test. Vì vậy, mọi mã trong cùng một nhóm dòng dõi được đưa vào **cùng một tập**. Mã không liên quan tới thu hồi tạo nhóm một phần tử, nên vẫn cùng tập với notebook 02.

**Cách xếp lớp** theo thứ tự ưu tiên: `NEW_ONLY` → `REVOKED_ANCHOR` → `ANSWER_CHANGED` → `LABEL_CHANGED` → `ANCHOR_RENAMED` → `STABLE`.

In [6]:
def split_of(anchor):                      # giống hệt notebook 02
    h = int(hashlib.sha1(f"{SEED}:{anchor}".encode()).hexdigest(), 16) % 100
    return "dev" if h < DEV_PERCENT else "test"

# ---- Chia dev/test theo "dòng dõi" (lineage) ------------------------------------------------
# Mã cũ bị thu hồi và (các) mã kế nhiệm của nó mang CÙNG một sự kiện (vd T1562.002 và T1685.001).
# Nếu chia theo từng mã riêng, cùng một sự kiện có thể rơi vào cả dev lẫn test (rò rỉ).
# Vì vậy ta gom mã cũ + mã kế nhiệm thành một nhóm, và băm theo mã nhỏ nhất của nhóm.
# Mã không dính tới thu hồi tạo nhóm một phần tử, nên vẫn trùng tập với notebook 02.
_parent = {}
def _find(x):
    _parent.setdefault(x, x)
    while _parent[x] != x:
        _parent[x] = _parent[_parent[x]]
        x = _parent[x]
    return x
for i, (succ, src) in SUCC.items():
    for s in succ:
        if s != i:
            a, b = _find(i), _find(s)
            if a != b:
                _parent[max(a, b)] = min(a, b)
LINEAGE = collections.defaultdict(set)
for x in list(_parent):
    LINEAGE[_find(x)].add(x)
def lineage_key(i):
    return min(LINEAGE[_find(i)]) if i in _parent else i
big = {k: sorted(v) for k, v in LINEAGE.items() if len(v) > 1}
print(f"{len(big)} nhóm dòng dõi nhiều mã (mã cũ ↔ mã kế nhiệm), ví dụ:", list(big.items())[:3])

# ---- Nhãn kiểu thay đổi LIÊN QUAN tới từng mẫu câu hỏi -----------------------------------------
# Một kỹ thuật đổi tactic không làm đổi đáp án D2 (kỹ thuật cha) hay D3 (mitigation),
# nên mỗi mẫu chỉ giữ các nhãn có thể làm đổi đáp án hoặc cách gọi tên của nó.
LINEAGE_EVENTS = {"ADDED", "REVOKED→NEW", "REVOKED→EXISTING", "MERGE", "DEPRECATED"}
RELEVANT = {"D1_tech_tactic":     {"TACTIC_CHANGED", "TACTIC_REDEFINED", "RENAMED"} | LINEAGE_EVENTS,
            "D2_subtech_parent":  {"REPARENTED", "RENAMED"} | LINEAGE_EVENTS,
            "D3_tech_mitigation": {"MITIGATION_CHANGED", "RENAMED"} | LINEAGE_EVENTS,
            "D4_id_currency":     {"RENAMED"} | LINEAGE_EVENTS}

def tac_disp(S, ids):  return ", ".join(f"{S['tactics'][t][1]} ({t})" for t in ids)
def tech_disp(S, ids): return "; ".join(f"{i} {S['tech'][i]['name']}" for i in ids)
def mit_disp(S, ids):  return "; ".join(f"{m} {S['mit'][m]['name']}" for m in ids)

def change_types(anchor_old, anchor_new, tpl, gold_old, gold_new, disp_old, disp_new):
    # Trả về (nhãn liên quan tới đáp án của mẫu này, các sự kiện khác của thực thể — chỉ để tham khảo)
    ct = set()
    for i in [anchor_old] + list(anchor_new):
        if i:
            ct |= EV_BY_ID.get(i, set())
    if tpl == "D1_tech_tactic" and (set(gold_old) | set(gold_new)) & REDEFINED_TACTICS:
        ct.add("TACTIC_REDEFINED")
    if anchor_old is None:
        ct -= {"MAJOR_REVISION", "REFINED"}
    rel = ct & RELEVANT[tpl]
    if (anchor_old and tpl in ("D2_subtech_parent", "D3_tech_mitigation")
            and set(gold_old) == set(gold_new) and disp_old != disp_new):
        rel.add("GOLD_RENAMED")            # đáp án giữ mã nhưng thực thể đáp án (kỹ thuật cha / mitigation) đổi tên
    return sorted(rel), sorted(ct - rel)

def classify(it):
    if it["anchor_old"] is None:
        return "NEW_ONLY"
    if it["anchor_status_new"] != "active":
        return "REVOKED_ANCHOR"
    if set(it["gold_old"]) != set(it["gold_new"]):
        return "ANSWER_CHANGED"
    if it["gold_display_old"] != it["gold_display_new"]:
        return "LABEL_CHANGED"
    if it["name_old"] != it["name_new"]:
        return "ANCHOR_RENAMED"
    return "STABLE"

# ---- Mẫu câu hỏi (tiếng Anh, giống văn phong notebook 02) ---------------------------------
Q = {
 "D1_tech_tactic": dict(answer_type="tactic_set",
    unversioned="Which MITRE ATT&CK tactic(s) does the technique {name} ({id}) belong to?",
    current=("According to the current MITRE ATT&CK Enterprise release (v{vn}), which tactic(s) does the technique "
             "{name} ({id}) belong to? If that technique ID has been replaced, answer for its replacement(s)."),
    historical="In MITRE ATT&CK Enterprise v{vo}, which tactic(s) did the technique {name} ({id}) belong to?"),
 "D2_subtech_parent": dict(answer_type="technique_set",
    unversioned='What is the parent ATT&CK technique (give its T-ID) of the sub-technique named "{name}"?',
    current=('In the current MITRE ATT&CK Enterprise release (v{vn}), what is the parent technique (give its T-ID) of the '
             'sub-technique named "{name}"? If it has been renamed or moved, answer for its current version.'),
    historical='In MITRE ATT&CK Enterprise v{vo}, what was the parent technique (give its T-ID) of the sub-technique named "{name}"?'),
 "D3_tech_mitigation": dict(answer_type="mitigation_set",
    unversioned="Which MITRE ATT&CK mitigations (M-IDs) are listed for the technique {name} ({id})?",
    current=("According to the current MITRE ATT&CK Enterprise release (v{vn}), which mitigations (M-IDs) are listed "
             "for the technique {name} ({id})? If that technique ID has been replaced, answer for its replacement(s)."),
    historical="In MITRE ATT&CK Enterprise v{vo}, which mitigations (M-IDs) were listed for the technique {name} ({id})?"),
 "D4_id_currency": dict(answer_type="technique_set",
    unversioned=("What is the current MITRE ATT&CK technique ID for {name} ({id})? If the ID is still valid, repeat it; "
                 "if it has been revoked and replaced, give the replacement technique ID(s)."),
    current=("As of MITRE ATT&CK Enterprise v{vn}, is the technique ID {id} ({name}) still valid? If yes, repeat the ID; "
             "if it has been replaced, give the replacement technique ID(s)."),
    historical=("In MITRE ATT&CK Enterprise v{vo}, was {id} ({name}) a valid technique ID? If yes, repeat the ID; "
                "if it had been replaced, give the replacement technique ID(s)."),
 ),
}
SHORT = {"D1_tech_tactic": "D1", "D2_subtech_parent": "D2", "D3_tech_mitigation": "D3", "D4_id_currency": "D4"}

def make_item(tpl, anchor_old, anchor_new, name_old, name_new, ask_id, ask_name,
              gold_old, gold_new, disp_old, disp_new, ev_old, ev_new, succ_src):
    t = Q[tpl]
    fr = {"unversioned": t["unversioned"], "current": t["current"]}
    if anchor_old is not None:
        fr["historical"] = t["historical"]
    it = dict(item_id=f"{SHORT[tpl]}-{anchor_old or anchor_new[0]}", template=tpl, answer_type=t["answer_type"],
              anchor_old=anchor_old, anchor_new=list(anchor_new), successor_source=succ_src,
              anchor_status_new=(NEW["tech"].get(anchor_old, {}).get("status") if anchor_old else "active"),
              name_old=name_old, name_new=name_new,
              # tên thực thể được hỏi (mọi phiên bản): khi chấm, các tên này bị che đi trước khi dò tên tactic,
              # để "Exploitation for Defense Evasion" không bị hiểu là trả lời tactic "Defense Evasion"
              entity_names=sorted({n for n in [name_old, ask_name] + [NEW["tech"][x]["name"] for x in anchor_new if x in NEW["tech"]] if n}),
              gold_old=sorted(gold_old), gold_new=sorted(gold_new),
              gold_display_old=disp_old, gold_display_new=disp_new,
              removed=sorted(set(gold_old) - set(gold_new)), added=sorted(set(gold_new) - set(gold_old)),
              questions={k: v.format(name=ask_name, id=ask_id, vo=V_OLD, vn=V_NEW) for k, v in fr.items()},
              evidence={V_OLD: ev_old, V_NEW: ev_new},
              old_id_in_new_corpus=(anchor_old is None or mentions.get(anchor_old, 1) > 0),
              lineage=lineage_key(anchor_old or anchor_new[0]),
              split=split_of(lineage_key(anchor_old or anchor_new[0])))
    it["change_types"], it["other_events"] = change_types(anchor_old, anchor_new, tpl, gold_old, gold_new, disp_old, disp_new)
    it["pair_class"] = classify(it)
    # Có thể trả lời "lỗi thời" không? Chỉ khi đáp án cũ có phần tử không còn đúng ở bản mới
    it["answer_changed"] = bool(anchor_old) and set(gold_old) != set(gold_new)
    return it

ITEMS, EXCLUDED = [], collections.Counter()

15 nhóm dòng dõi nhiều mã (mã cũ ↔ mã kế nhiệm), ví dụ: [('T1070.001', ['T1070.001', 'T1685.005']), ('T1070.002', ['T1070.002', 'T1685.006']), ('T1562', ['T1562', 'T1562.001', 'T1562.006', 'T1685'])]


### 6.1. D1 — kỹ thuật thuộc tactic nào

Quần thể: mọi kỹ thuật còn hiệu lực ở v18.1, cộng các kỹ thuật **mới xuất hiện** ở v19.2 (`NEW_ONLY`). Với mã bị thu hồi, đáp án hiện hành là **hợp** các tactic của mọi mã kế nhiệm.

In [7]:
TPL = "D1_tech_tactic"
for i in sorted(ACT_OLD):
    succ, src = SUCC[i]
    if not succ:
        EXCLUDED[(TPL, "deprecated_no_successor")] += 1; continue
    go = list(OLD["tech"][i]["tactics"]); gn = sorted({t for s in succ for t in NEW["tech"][s]["tactics"]})
    if not go or not gn:
        EXCLUDED[(TPL, "no_tactic")] += 1; continue
    nn = NEW["tech"][succ[0]]["name"] if len(succ) == 1 else " / ".join(NEW["tech"][s]["name"] for s in succ)
    ITEMS.append(make_item(TPL, i, succ, OLD["tech"][i]["name"], nn, i, OLD["tech"][i]["name"],
                           go, gn, tac_disp(OLD, go), tac_disp(NEW, gn),
                           ev_tech(V_OLD, [i]), ev_tech(V_NEW, succ), src))
for i in sorted(ACT_NEW - ACT_OLD):
    gn = list(NEW["tech"][i]["tactics"])
    if not gn:
        EXCLUDED[(TPL, "no_tactic")] += 1; continue
    ITEMS.append(make_item(TPL, None, [i], None, NEW["tech"][i]["name"], i, NEW["tech"][i]["name"],
                           [], gn, "", tac_disp(NEW, gn), [], ev_tech(V_NEW, [i]), "new"))
print(collections.Counter(it["pair_class"] for it in ITEMS if it["template"] == TPL))

Counter({'STABLE': 475, 'LABEL_CHANGED': 126, 'ANSWER_CHANGED': 72, 'NEW_ONLY': 23, 'REVOKED_ANCHOR': 17, 'ANCHOR_RENAMED': 1})


### 6.2. D2 — sub-technique (hỏi bằng tên) có kỹ thuật cha nào

Như notebook 02, câu hỏi dùng **tên** chứ không dùng mã, vì mã sub-technique (T1001.003) đã lộ mã cha. Chỉ giữ tên **duy nhất** trong phiên bản tương ứng.

Câu hỏi yêu cầu trả lời bằng **mã T** để chấm được bằng mã.

Hai trường hợp đặc biệt:

- `PROMOTED`: sub-technique bị thu hồi và kế nhiệm là một **kỹ thuật cấp cha**. Ví dụ T1562.003 → T1690, T1562.001 → T1685: ở bản mới không còn "kỹ thuật cha" để hỏi. Các mục này **bị loại khỏi D2** nhưng vẫn có mặt trong D1, D3, D4.
- Chiều ngược lại: **kỹ thuật cấp cha trở thành sub-technique**. Ví dụ T1656 *Impersonation* → T1684.001, T1672 *Email Spoofing* → T1684.002. Ở bản cũ chúng không có kỹ thuật cha, nên D2 chỉ hỏi được ở bản mới. Chúng được xếp vào `NEW_ONLY` với nhãn `REVOKED→NEW`. Sub-technique mới chỉ bị loại khi **cùng tên đã được hỏi** qua một mục D2 của mã cũ (ví dụ *Clear Windows Event Logs* đã được hỏi qua T1070.001).

In [8]:
TPL = "D2_subtech_parent"
def unique_names(S, ids):
    c = collections.Counter(S["tech"][i]["name"].lower() for i in ids)
    return {n for n, k in c.items() if k == 1}
UNIQ_OLD, UNIQ_NEW = unique_names(OLD, ACT_OLD), unique_names(NEW, ACT_NEW)
D2_ASKED = set()                           # tên đã được hỏi qua một mục D2 của mã cũ

for i in sorted(x for x in ACT_OLD if OLD["tech"][x]["is_sub"]):
    succ, src = SUCC[i]
    name = OLD["tech"][i]["name"]
    if name.lower() not in UNIQ_OLD:
        EXCLUDED[(TPL, "name_not_unique")] += 1; continue
    if not succ:
        EXCLUDED[(TPL, "deprecated_no_successor")] += 1; continue
    if not all(NEW["tech"][s]["is_sub"] for s in succ):
        EXCLUDED[(TPL, "PROMOTED")] += 1; continue
    go = [OLD["tech"][i]["parent"]]; gn = sorted({NEW["tech"][s]["parent"] for s in succ})
    nn = " / ".join(NEW["tech"][s]["name"] for s in succ)
    ITEMS.append(make_item(TPL, i, succ, name, nn, i, name, go, gn, tech_disp(OLD, go), tech_disp(NEW, gn),
                           ev_tech(V_OLD, [i]), ev_tech(V_NEW, succ), src))
    D2_ASKED.add(name.lower())
for i in sorted(x for x in ACT_NEW - ACT_OLD if NEW["tech"][x]["is_sub"]):
    name = NEW["tech"][i]["name"]
    if name.lower() in D2_ASKED:           # cùng tên đã được hỏi qua một mục D2 của mã cũ (vd REVOKED_ANCHOR)
        EXCLUDED[(TPL, "already_asked_by_old_name")] += 1; continue
    if name.lower() not in UNIQ_NEW:
        EXCLUDED[(TPL, "name_not_unique")] += 1; continue
    gn = [NEW["tech"][i]["parent"]]
    ITEMS.append(make_item(TPL, None, [i], None, name, i, name, [], gn, "", tech_disp(NEW, gn),
                           [], ev_tech(V_NEW, [i]), "new"))
print(collections.Counter(it["pair_class"] for it in ITEMS if it["template"] == TPL))

Counter({'STABLE': 410, 'NEW_ONLY': 12, 'REVOKED_ANCHOR': 8, 'ANCHOR_RENAMED': 3})


### 6.3. D3 — kỹ thuật có những biện pháp giảm thiểu nào

Chỉ giữ mục có từ 1 đến `MAX_MITIGATIONS` mitigation ở **cả hai** phiên bản (với `NEW_ONLY`: ở bản mới).

In [9]:
TPL = "D3_tech_mitigation"
ok = lambda s: 1 <= len(s) <= MAX_MITIGATIONS
for i in sorted(ACT_OLD):
    succ, src = SUCC[i]
    if not succ:
        EXCLUDED[(TPL, "deprecated_no_successor")] += 1; continue
    go = sorted(OLD["mits_of"].get(i, set())); gn = sorted({m for s in succ for m in NEW["mits_of"].get(s, set())})
    if not (ok(go) and ok(gn)):
        EXCLUDED[(TPL, "mitigation_count_out_of_range")] += 1; continue
    nn = " / ".join(NEW["tech"][s]["name"] for s in succ)
    ITEMS.append(make_item(TPL, i, succ, OLD["tech"][i]["name"], nn, i, OLD["tech"][i]["name"],
                           go, gn, mit_disp(OLD, go), mit_disp(NEW, gn),
                           ev_mit(V_OLD, [i], go), ev_mit(V_NEW, succ, gn), src))
for i in sorted(ACT_NEW - ACT_OLD):
    gn = sorted(NEW["mits_of"].get(i, set()))
    if not ok(gn):
        EXCLUDED[(TPL, "mitigation_count_out_of_range")] += 1; continue
    ITEMS.append(make_item(TPL, None, [i], None, NEW["tech"][i]["name"], i, NEW["tech"][i]["name"],
                           [], gn, "", mit_disp(NEW, gn), [], ev_mit(V_NEW, [i], gn), "new"))
print(collections.Counter(it["pair_class"] for it in ITEMS if it["template"] == TPL))

Counter({'STABLE': 539, 'NEW_ONLY': 20, 'REVOKED_ANCHOR': 14, 'ANCHOR_RENAMED': 4})


### 6.4. D4 — mã kỹ thuật còn hiệu lực không

Ở bản cũ, đáp án luôn là chính mã đó. Ở bản mới, đáp án là (các) mã kế nhiệm. Với mã bị thu hồi, bằng chứng trong kho bản mới **để trống**, vì kho không còn chứa liên kết mã cũ → mã mới (xem mục 5). Mẫu này không có `NEW_ONLY`.

In [10]:
TPL = "D4_id_currency"
for i in sorted(ACT_OLD):
    succ, src = SUCC[i]
    if not succ:
        EXCLUDED[(TPL, "deprecated_no_successor")] += 1; continue
    nn = " / ".join(NEW["tech"][s]["name"] for s in succ)
    ITEMS.append(make_item(TPL, i, succ, OLD["tech"][i]["name"], nn, i, OLD["tech"][i]["name"],
                           [i], succ, tech_disp(OLD, [i]), tech_disp(NEW, succ),
                           ev_tech(V_OLD, [i]), ev_tech(V_NEW, [i]) if src == "same" else [], src))
# Với D4, đổi tên thực thể không làm đổi đáp án (mã) — xếp vào ANCHOR_RENAMED thay vì LABEL_CHANGED
for it in ITEMS:
    if it["template"] == TPL and it["pair_class"] == "LABEL_CHANGED":
        it["pair_class"] = "ANCHOR_RENAMED"
print(collections.Counter(it["pair_class"] for it in ITEMS if it["template"] == TPL))

Counter({'STABLE': 670, 'REVOKED_ANCHOR': 17, 'ANCHOR_RENAMED': 4})


## 7. Tổng hợp quần thể

Bảng 1 là **số mục theo mẫu × lớp cặp**, dùng cho bảng thống kê bộ dữ liệu trong bài. Bảng 2 cho biết mỗi lớp ứng với những kiểu thay đổi nào của notebook 01.

In [11]:
CLASSES = ["STABLE", "ANSWER_CHANGED", "LABEL_CHANGED", "ANCHOR_RENAMED", "REVOKED_ANCHOR", "NEW_ONLY"]
df = pd.DataFrame(ITEMS)
tab = df.pivot_table(index="template", columns="pair_class", values="item_id", aggfunc="count", fill_value=0)
tab = tab.reindex(columns=[c for c in CLASSES if c in tab.columns])
tab["TỔNG"] = tab.sum(axis=1)
tab.loc["TỔNG"] = tab.sum()
tab.to_csv(OUT / "population_by_class.csv")
print("Loại khỏi quần thể:", dict(EXCLUDED))
tab

Loại khỏi quần thể: {('D2_subtech_parent', 'name_not_unique'): 48, ('D2_subtech_parent', 'PROMOTED'): 6, ('D2_subtech_parent', 'already_asked_by_old_name'): 2, ('D3_tech_mitigation', 'mitigation_count_out_of_range'): 137}


pair_class,STABLE,ANSWER_CHANGED,LABEL_CHANGED,ANCHOR_RENAMED,REVOKED_ANCHOR,NEW_ONLY,TỔNG
template,,,,,,,
D1_tech_tactic,475,72,126,1,17,23,714
D2_subtech_parent,410,0,0,3,8,12,433
D3_tech_mitigation,539,0,0,4,14,20,577
D4_id_currency,670,0,0,4,17,0,691
TỔNG,2094,72,126,12,56,55,2415


In [12]:
ct = df[df.pair_class != "STABLE"].explode("change_types")
ct = ct.pivot_table(index="pair_class", columns="change_types", values="item_id", aggfunc="count", fill_value=0)
ct

change_types,ADDED,MERGE,RENAMED,REVOKED→NEW,TACTIC_CHANGED,TACTIC_REDEFINED
pair_class,,,,,,
ANCHOR_RENAMED,0,0,12,0,0,0
ANSWER_CHANGED,0,0,2,0,72,72
LABEL_CHANGED,0,0,1,0,0,126
NEW_ONLY,55,1,0,37,0,4
REVOKED_ANCHOR,56,6,0,56,0,17


In [13]:
# Xem nhanh vài mục tiêu biểu
show = ["item_id", "pair_class", "change_types", "gold_display_old", "gold_display_new"]
pick = [f"D1-T1222", f"D1-T1027", f"D1-T1562", f"D2-T1070.001", f"D3-T1562.004", f"D4-T1562.004", f"D4-T1001"]
df[df.item_id.isin(pick)][show]

,item_id,pair_class,change_types,gold_display_old,gold_display_new
38,D1-T1027,LABEL_CHANGED,[TACTIC_REDEFINED],Defense Evasion (TA0005),Stealth (TA0005)
299,D1-T1222,ANSWER_CHANGED,"[TACTIC_CHANGED, TACTIC_REDEFINED]",Defense Evasion (TA0005),Defense Impairment (TA0112)
470,D1-T1562,REVOKED_ANCHOR,"[ADDED, MERGE, REVOKED→NEW, TACTIC_REDEFINED]",Defense Evasion (TA0005),Defense Impairment (TA0112)
810,D2-T1070.001,REVOKED_ANCHOR,"[ADDED, REVOKED→NEW]",T1070 Indicator Removal,T1685 Disable or Modify Tools
1514,D3-T1562.004,REVOKED_ANCHOR,"[ADDED, REVOKED→NEW]",M1018 User Account Management; M1022 Restrict ...,M1018 User Account Management; M1022 Restrict ...
1724,D4-T1001,STABLE,[],T1001 Data Obfuscation,T1001 Data Obfuscation
2198,D4-T1562.004,REVOKED_ANCHOR,"[ADDED, REVOKED→NEW]",T1562.004 Disable or Modify System Firewall,T1686 Disable or Modify System Firewall


## 8. Kiểm tra tự động

| # | Kiểm tra | Vì sao |
|---|---|---|
| 1 | `item_id` không trùng | khoá chính |
| 2 | Mọi mã trong `gold_old` còn hiệu lực ở v18.1, mọi mã trong `gold_new` còn hiệu lực ở v19.2 | đáp án đúng **theo đúng phiên bản** |
| 3 | `STABLE` ⇔ đáp án, nhãn hiển thị, tên và trạng thái giống hệt; mọi lớp khác có ít nhất một nhãn thay đổi liên quan | nhãn nhất quán với ground truth |
| 4 | Mọi mục D1 `ANSWER_CHANGED` (mã giữ nguyên) đều có sự kiện `TACTIC_CHANGED` ở notebook 01 (với cặp phiên bản liền kề như 18.1→19.2: **trùng khớp hoàn toàn**) | đối chiếu chéo hai cách tính độc lập |
| 5 | Mọi đoạn bằng chứng tồn tại trong kho của phiên bản đó **và chứa mã đáp án** | bằng chứng thật sự chứa đáp án |
| 6 | Mục có thực thể không liên quan tới thu hồi thì cùng tập dev/test với notebook 02 | nhất quán giữa các notebook |
| 7 | Các mục cùng mẫu có cùng mã thực thể hiện hành (ví dụ D1-T1562.002 và D1-T1685.001) luôn nằm cùng một tập | không rò rỉ giữa dev và test |

In [14]:
CHECKS = {}
# 1
CHECKS["1_unique_item_id"] = df.item_id.is_unique
# 2
valid = {V_OLD: {"tactic_set": set(OLD["tactics"]), "technique_set": ACT_OLD, "mitigation_set": {m for m, x in OLD["mit"].items() if x["status"] == "active"}},
         V_NEW: {"tactic_set": set(NEW["tactics"]), "technique_set": ACT_NEW, "mitigation_set": {m for m, x in NEW["mit"].items() if x["status"] == "active"}}}
bad2 = [it["item_id"] for it in ITEMS
        if not set(it["gold_old"]) <= valid[V_OLD][it["answer_type"]] or not set(it["gold_new"]) <= valid[V_NEW][it["answer_type"]]
        or not it["gold_new"] or (it["anchor_old"] and not it["gold_old"])]
CHECKS["2_gold_valid_in_version"] = not bad2
# 3
bad3 = [it["item_id"] for it in ITEMS if
        (it["pair_class"] == "STABLE") != (it["gold_old"] == it["gold_new"] and it["gold_display_old"] == it["gold_display_new"]
                                           and it["name_old"] == it["name_new"] and it["anchor_status_new"] == "active")
        or (it["pair_class"] != "STABLE" and not it["change_types"])]
CHECKS["3_class_label_consistent"] = not bad3
stable_labeled = [it["item_id"] for it in ITEMS if it["pair_class"] == "STABLE" and it["change_types"]]
if stable_labeled:   # chỉ có thể xảy ra khi khoảng so sánh dài và thay đổi bị đảo ngược ở bản sau
    print(f"ℹ {len(stable_labeled)} mục STABLE vẫn mang nhãn thay đổi (thay đổi bị đảo ngược trong khoảng):", stable_labeled[:10])
# 4
d1_changed = set(df[(df.template == "D1_tech_tactic") & (df.pair_class == "ANSWER_CHANGED")].anchor_old)
nb01_tc = set(EV[EV.type == "TACTIC_CHANGED"].id_old) & ACT_OLD & ACT_NEW
CHECKS["4_D1_backed_by_nb01_TACTIC_CHANGED"] = d1_changed <= nb01_tc
reverted = sorted(nb01_tc - d1_changed)
print(f"D1 ANSWER_CHANGED: {len(d1_changed)} mục | TACTIC_CHANGED của notebook 01: {len(nb01_tc)} mã"
      + (f" | đổi rồi đổi lại trong khoảng: {reverted[:10]}" if reverted else " | trùng khớp hoàn toàn"))
# 5
def evidence_ok(it, v):
    gold = it["gold_old"] if v == V_OLD else it["gold_new"]
    ids = it["evidence"][v]
    if not ids:
        return True
    if any(c not in CORPUS[v] for c in ids):
        return False
    text = "\n".join(CORPUS[v][c]["text"] for c in ids)
    return all(re.search(re.escape(g) + r"(?![.\d])", text) for g in gold)
bad5 = [(it["item_id"], v) for it in ITEMS for v in (V_OLD, V_NEW) if not evidence_ok(it, v)]
CHECKS["5_evidence_contains_gold"] = not bad5
no_ev = collections.Counter((it["template"], it["pair_class"]) for it in ITEMS
                            if it["anchor_old"] and not it["evidence"][V_OLD])
# 6
TPL_NB02 = {"D1_tech_tactic": "Q1_tech_tactic", "D2_subtech_parent": "Q2_subtech_parent", "D3_tech_mitigation": "Q3_tech_mitigation"}
bad6, n6 = [], 0
for v in (V_OLD, V_NEW):
    qp = ROOT / "benchmark" / f"attack-{v}" / "questions.jsonl"
    if qp.exists():
        nb02 = {(q["template"], q["anchor"]): q["split"] for q in map(json.loads, open(qp, encoding="utf-8"))}
        for it in ITEMS:
            a = it["anchor_old"] or it["anchor_new"][0]
            k = (TPL_NB02.get(it["template"]), a)
            if k in nb02 and a not in _parent:      # thực thể không thuộc nhóm dòng dõi nào
                n6 += 1
                if nb02[k] != it["split"]:
                    bad6.append(it["item_id"])
CHECKS["6_split_consistent_with_nb02"] = not bad6
print(f"Kiểm tra 6: so {n6} cặp (mục, câu hỏi notebook 02) có thực thể không liên quan tới thu hồi")
# 7
seen7, bad7 = {}, []
for it in ITEMS:
    for x in it["anchor_new"]:
        k = (it["template"], x)
        if k in seen7 and seen7[k] != it["split"]:
            bad7.append(it["item_id"])
        seen7.setdefault(k, it["split"])
CHECKS["7_no_dev_test_leak_via_successor"] = not bad7

for k, v in CHECKS.items():
    print(("✓" if v else "✗"), k)
print("Mục cũ không có bằng chứng ở kho bản cũ:", dict(no_ev) or "không có")
assert all(CHECKS.values()), dict(bad2=bad2[:10], bad3=bad3[:10], d1_not_in_nb01=sorted(d1_changed - nb01_tc)[:10],
                                  bad5=bad5[:10], bad6=bad6[:10], bad7=bad7[:10])
MANIFEST["checks"] = CHECKS

D1 ANSWER_CHANGED: 72 mục | TACTIC_CHANGED của notebook 01: 72 mã | trùng khớp hoàn toàn
Kiểm tra 6: so 2342 cặp (mục, câu hỏi notebook 02) có thực thể không liên quan tới thu hồi
✓ 1_unique_item_id
✓ 2_gold_valid_in_version
✓ 3_class_label_consistent
✓ 4_D1_backed_by_nb01_TACTIC_CHANGED
✓ 5_evidence_contains_gold
✓ 6_split_consistent_with_nb02
✓ 7_no_dev_test_leak_via_successor
Mục cũ không có bằng chứng ở kho bản cũ: không có


## 9. Module chấm điểm theo phiên bản `ctidrift_scoring.py`

Module này được ghi ra file để notebook 04 và 05 `import` lại. Toàn bộ bài báo dùng **một định nghĩa chỉ số duy nhất**.

**Định dạng trả lời chính: JSON.** Ở notebook 04, mô hình được **ép** trả lời theo lược đồ `{"answer": ["T1686", ...]}` bằng *guided decoding* của vLLM. Đây là cách làm chuẩn để chấm tự động mà không phải đoán ý câu văn. Module đọc đối tượng JSON **cuối cùng** có khoá `answer` và **chỉ** đọc khoá đó; các khoá khác như `reasoning` không được chấm. Nếu có JSON hợp lệ, JSON luôn được ưu tiên hơn dòng `ANSWER:`. Kết quả chấm có thêm trường `format` (`json` / `marker` / `free`) để báo cáo **tỉ lệ vi phạm định dạng**, chẳng hạn khi chạy mô hình qua API không hỗ trợ ép định dạng.

**Phương án dự phòng (không có JSON):** mô hình được yêu cầu kết thúc bằng một dòng `ANSWER: ...` **chỉ chứa đáp án**. Nếu câu trả lời có dòng này, ta chỉ chấm **phần trả lời**:

- Phần trả lời là phần còn lại của dòng `ANSWER:`, cộng các dòng ngay sau **nếu đó là mục danh sách**. Mục danh sách là dòng bắt đầu bằng một mã (có thể có gạch đầu dòng), theo sau chỉ là tên viết hoa, ví dụ `M1041 Encrypt Sensitive Information`. Dòng chỉ gồm tên tactic cũng được tính. Các dòng giải thích (câu văn) phía sau **không được tính**, kể cả khi chứa mã, ví dụ "T1562.004 is the old ID."
- Mục có gạch đầu dòng bắt đầu bằng mã luôn được tính, kể cả khi mô tả viết thường ("- M1029: store logs remotely").
- Nếu dòng `ANSWER:` không chứa đáp án (trống, hoặc chỉ là câu dẫn như "The following mitigations apply:"), lấy **cả đoạn kế tiếp** (danh sách, bảng, chữ đậm, "tên (mã)"…).
- Khi có nhiều dòng `ANSWER:`, lấy dòng **cuối**. Nếu dòng cuối là **từ chối trả lời** ("not sure", "unknown"…) thì tính là không trả lời. Nếu dòng cuối không chứa đáp án (ví dụ "ANSWER: done") thì lùi về dòng `ANSWER:` gần nhất có đáp án.

Lý do có quy ước này: mô hình hay giải thích kiểu *"T1562.004 đã được thay bằng T1686"*. Nếu chấm cả câu thì mã cũ bị tính là câu trả lời.

**Các quy tắc trích xuất** (được kiểm thử ở mục 9.2):

- Mã không phân biệt hoa/thường (`t1686` = `T1686`).
- Tên tactic chỉ được dùng để suy ra mã khi phần trả lời **không có mã TA nào**. Nếu mô hình đã ghi mã, tên chỉ dùng để phát hiện nhãn lỗi thời.
- Tên tactic viết thường ("defense impairment") chỉ được nhận khi phần trả lời là **danh sách thuần** (chỉ gồm tên, mã, dấu câu). Khi có lẫn câu văn, tên phải viết đúng như MITRE (*Defense Evasion*) hoặc viết hoa toàn bộ. Nhờ vậy các chữ "impact", "stealth", "initial access" trong câu văn không bị hiểu nhầm là tên tactic. Dấu gạch nối không được coi là ranh giới từ ("pre-execution" không khớp *Execution*).
- **Tên của chính thực thể được hỏi bị che đi (theo trọn từ)** trước khi dò tên tactic. Ví dụ "Exploitation for Defense Evasion" hay "Indirect Command Execution" không bị hiểu là tactic.
- Một câu như *"T1686 (formerly T1562.004)"* **trên dòng ANSWER** vẫn bị chấm `STALE`. Đây là quy ước có chủ đích: dòng ANSWER chỉ được chứa đáp án, và prompt sẽ nói rõ điều đó.

**Giới hạn đã biết của phương án dự phòng** (được người kiểm tra độc lập xác nhận, chấp nhận vì định dạng chính là JSON):

- Tên tactic viết thường **trong câu văn** ("It belongs to the defense impairment tactic") không được nhận.
- Dòng sau ANSWER bắt đầu bằng mã cũ kèm tên viết hoa ("T1562.004 Disable Or Modify…") bị tính là mục danh sách.
- "command-and-control", "C2" không được nhận.
- Văn bản tự do không có ANSWER, nhắc cả đáp án cũ lẫn mới, bị chấm `STALE`.

Ở notebook 04, ta **báo cáo tỉ lệ `format != "json"`**. Nếu tỉ lệ này đáng kể, phải chấm tay một mẫu để ước lượng sai số của phương án dự phòng.

Với mục `STABLE`, *bên kia* trùng *đích*, nên `STALE` không thể xảy ra. Nhóm này cho ta mức lỗi nền để so sánh.

In [15]:
%%writefile ctidrift_scoring.py
# Chấm điểm câu trả lời theo phiên bản (notebook 03, v1.1). Dùng chung cho notebook 04, 05.
from __future__ import annotations
import json
import re

ID_RE = {
    "technique_set":  re.compile(r"\bT\d{4}(?:\.\d{3})?\b", re.I),
    "tactic_set":     re.compile(r"\bTA\d{4}\b", re.I),
    "mitigation_set": re.compile(r"\bM\d{4}\b", re.I),
}
ANY_ID = re.compile(r"\b(?:TA|T|M)\d{4}(?:\.\d{3})?\b", re.I)
MARKER_RE = re.compile(r"^[ \t>*_#-]*(?:final[ \t]+)?answer[ \t*_]*[:：][ \t*_]*", re.I | re.M)
BULLET_RE = re.compile(r"^\s*(?:[-*•+]|\d+[.)])\s+")

ABSTAIN_RE = re.compile(r"\b(?:not\s+sure|unsure|unknown|don'?t\s+know|do\s+not\s+know|cannot\s+(?:determine|answer)|"
                        r"can'?t\s+(?:determine|answer)|no\s+answer|n/?a|none)\b", re.I)

CONNECTORS = {"and", "or", "of", "for", "the", "to", "a", "an", "in", "on", "with", "via", "from", "by", "at"}
LIST_FILLER = {"and", "or", "the", "tactic", "tactics", "both"}

def _name_pattern(name: str, ignore_case: bool):
    # Gạch nối không phải ranh giới từ ("pre-execution" không khớp Execution).
    body = re.escape(name).replace(r"\ ", r"\s+")
    if ignore_case:
        return re.compile(r"(?<![\w-])" + body + r"(?![\w-])", re.I)
    return re.compile(r"(?<![\w-])(?:" + body + "|" + re.escape(name.upper()).replace(r"\ ", r"\s+") + r")(?![\w-])")

def _mask(text: str, tactic_alias: dict, mask=()) -> str:
    t = text
    official = {n.lower() for n in (tactic_alias or {})}
    for m in sorted(mask, key=len, reverse=True):          # che tên thực thể được hỏi (trọn từ)
        if m and m.lower() not in official:
            t = re.sub(r"(?<![\w-])" + re.escape(m) + r"(?![\w-])", " ", t, flags=re.I)
    return re.sub(r"\s*&\s*", " and ", t)                # "Command & Control"

def _is_pure_list(text: str, tactic_alias: dict) -> bool:
    # Đúng nếu đoạn chỉ gồm mã, tên tactic (hoa/thường tuỳ ý), dấu câu và vài từ nối — không có câu văn
    t = ANY_ID.sub(" ", text)
    for n in sorted(tactic_alias, key=len, reverse=True):
        t = _name_pattern(n, True).sub(" ", t)
    words = re.sub(r"[^\w\s]", " ", t).lower().split()
    return all(w in LIST_FILLER for w in words)

def _tactic_names(text: str, tactic_alias: dict, mask=(), lenient: bool = False) -> dict:
    # lenient=True (chỉ dùng cho dòng ANSWER là danh sách thuần): nhận tên viết hoa/thường tuỳ ý.
    # Còn lại: chỉ đúng cách viết của MITRE hoặc viết hoa toàn bộ, để chữ "impact", "stealth",
    # "initial access" trong câu văn không bị hiểu là tên tactic.
    t = _mask(text, tactic_alias, mask)
    ignore_case = lenient and _is_pure_list(t, tactic_alias)
    return {n: tid for n, tid in tactic_alias.items() if _name_pattern(n, ignore_case).search(t)}

def _has_answer(seg: str, answer_type, tactic_alias, mask) -> bool:
    if answer_type is None:
        return bool(seg.strip())
    if ID_RE[answer_type].search(seg):
        return True
    return answer_type == "tactic_set" and bool(tactic_alias) and bool(_tactic_names(seg, tactic_alias, mask, True))

def _is_list_item(line: str, answer_type, tactic_alias) -> bool:
    # Một dòng là "mục danh sách" nếu (sau gạch đầu dòng) bắt đầu bằng một mã và phần còn lại chỉ là
    # tên viết hoa (vd "M1041 Encrypt Sensitive Information"), hoặc là danh sách thuần các tên tactic.
    bullet = bool(BULLET_RE.match(line))
    body = re.sub(r"^[*_`]+", "", BULLET_RE.sub("", line, count=1).strip())
    m = ANY_ID.match(body)
    if m:
        if bullet:                                         # "- M1029: store logs remotely" vẫn là một mục
            return True
        words = re.findall(r"[A-Za-z][\w'/-]*", ANY_ID.sub(" ", body[m.end():]))
        return all(w[0].isupper() or w.lower() in CONNECTORS for w in words)
    return answer_type == "tactic_set" and bool(tactic_alias) and bool(body) and _is_pure_list(body, tactic_alias)

def _segment(text_after_marker: str, answer_type=None, tactic_alias=None, mask=()) -> str:
    # Phần trả lời = phần còn lại của dòng ANSWER, cộng các dòng ngay sau nếu là MỤC DANH SÁCH.
    # Nếu dòng ANSWER không chứa đáp án (trống, hoặc chỉ là câu dẫn "The following mitigations apply:"),
    # bỏ qua dòng trống và lấy các mục danh sách của đoạn kế tiếp.
    lines = text_after_marker.split("\n")
    first, rest = lines[0], lines[1:]
    seg, i = [first], 0
    if not _has_answer(first, answer_type, tactic_alias, mask):
        while i < len(rest) and not rest[i].strip():
            i += 1
        para = []
        while i < len(rest) and rest[i].strip() and not MARKER_RE.match(rest[i]):
            para.append(rest[i]); i += 1
        # Dòng ANSWER chỉ là câu dẫn: lấy CẢ đoạn kế tiếp (bảng, chữ đậm, "tên (mã)"…)
        return "\n".join([first] + para)
    while i < len(rest) and rest[i].strip() and not MARKER_RE.match(rest[i]) \
            and _is_list_item(rest[i], answer_type, tactic_alias):
        seg.append(rest[i]); i += 1
    return "\n".join(seg)

def _json_objects(text: str) -> list:
    # Mọi đối tượng JSON cấp ngoài cùng trong văn bản (giải mã bằng json.JSONDecoder, chịu được JSON lồng nhau,
    # dấu ngoặc trong chuỗi, khối ```json ... ```)
    dec, out, i = json.JSONDecoder(), [], 0
    while True:
        j = text.find("{", i)
        if j < 0:
            return out
        try:
            obj, end = dec.raw_decode(text, j)
        except ValueError:
            i = j + 1
            continue
        if isinstance(obj, dict):
            out.append(obj)
        i = end

def json_answer(answer: str):
    """Định dạng CHÍNH (notebook 04 dùng guided decoding): {"answer": ["T1686", ...]}.
    Lấy đối tượng JSON CUỐI CÙNG có khoá "answer" (không phân biệt hoa/thường) và CHỈ đọc khoá đó —
    các khoá khác (reasoning, note…) không được chấm. Trả về list chuỗi, hoặc None nếu không có JSON hợp lệ."""
    for obj in reversed(_json_objects(answer or "")):
        low = {str(k).lower(): v for k, v in obj.items()}
        if "answer" not in low:
            continue
        v = low["answer"]
        if v is None:
            return []
        if isinstance(v, (str, int, float)):
            return [str(v)]
        if isinstance(v, list):
            items = []
            for x in v:
                if isinstance(x, dict):                    # [{"id": "T1686", ...}] -> chỉ lấy "id"
                    xl = {str(k).lower(): y for k, y in x.items()}
                    if "id" in xl:
                        items.append(str(xl["id"]))
                elif x is not None:
                    items.append(str(x))
            return items
        return []
    return None

def final_part(answer: str, answer_type: str | None = None, tactic_alias: dict | None = None, mask=()):
    """Trả về (phần trả lời, có_dòng_ANSWER). Không có dòng ANSWER thì trả lại toàn bộ câu trả lời.
    Có nhiều dòng ANSWER: lấy dòng cuối; nếu dòng cuối là từ chối trả lời ("not sure") thì coi là không trả lời;
    nếu dòng cuối không chứa đáp án (vd "ANSWER: done") thì lùi về dòng ANSWER gần nhất có đáp án."""
    a = answer or ""
    marks = list(MARKER_RE.finditer(a))
    if not marks:
        return a, False
    segs = [_segment(a[m.end():], answer_type, tactic_alias, mask) for m in marks]
    last = segs[-1]
    if _has_answer(last, answer_type, tactic_alias, mask) or ABSTAIN_RE.search(last):
        return last, True
    for sg in reversed(segs[:-1]):
        if _has_answer(sg, answer_type, tactic_alias, mask):
            return sg, True
    return last, True

def extract(answer: str, answer_type: str, tactic_alias: dict | None = None, mask=()):
    """Trả về (tập mã, tập tên tactic đã dùng). tactic_alias: tên chính thức -> mã TA, gồm tên của CẢ HAI phiên bản."""
    ja = json_answer(answer)
    if ja is not None:                                     # định dạng chính: JSON
        a, marked = "\n".join(ja), True
    else:                                                  # dự phòng: dòng ANSWER hoặc văn bản tự do
        a, marked = final_part(answer, answer_type, tactic_alias, mask)
    ids = {x.upper() for x in ID_RE[answer_type].findall(a)}
    names = set()
    if answer_type == "tactic_set" and tactic_alias:
        found = _tactic_names(a, tactic_alias, mask, lenient=marked)
        names = set(found)
        if not ids:                                        # chỉ suy mã từ tên khi không có mã TA nào
            ids = set(found.values())
    return ids, names

def score_drift(q: dict, answer: str, tactic_alias: dict | None = None) -> dict:
    """q là một dòng của queries_*.jsonl (gold, other_gold, direction, other_only_labels, entity_names)."""
    pred, names = extract(answer, q["answer_type"], tactic_alias, q.get("entity_names", ()))
    fmt = "json" if json_answer(answer) is not None else ("marker" if MARKER_RE.search(answer or "") else "free")
    target, other = set(q["gold"]), set(q["other_gold"])
    tp = len(pred & target)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(target) if target else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    drift = pred & (other - target)
    if pred == target:
        outcome = "CORRECT"
    elif drift:
        outcome = "STALE" if q["direction"] == "current" else "ANACHRONISM"
    elif tp:
        outcome = "PARTIAL"
    elif not pred:
        outcome = "NO_ANSWER"
    else:
        outcome = "WRONG"
    return dict(pred=sorted(pred), em=pred == target, precision=p, recall=r, f1=f1,
                drift_hit=bool(drift), drift_ids=sorted(drift), drift_possible=bool(other - target),
                label_drift=bool(names & set(q.get("other_only_labels", []))), outcome=outcome, format=fmt)

Writing ctidrift_scoring.py


In [16]:
import importlib, ctidrift_scoring
importlib.reload(ctidrift_scoring)
from ctidrift_scoring import score_drift, extract, final_part

# Bảng tên tactic (đúng cách viết của MITRE) của CẢ HAI phiên bản -> mã.
# Tên cũ "Defense Evasion" và tên mới "Stealth" đều -> TA0005.
TACTIC_ALIAS = {}
for S in (OLD, NEW):
    for tid, (_, name) in S["tactics"].items():
        assert TACTIC_ALIAS.get(name, tid) == tid, f"Tên tactic {name} trỏ tới hai mã khác nhau"
        TACTIC_ALIAS[name] = tid
names_old = {n for _, n in OLD["tactics"].values()}; names_new = {n for _, n in NEW["tactics"].values()}
OLD_ONLY_LABELS = sorted(names_old - names_new)
NEW_ONLY_LABELS = sorted(names_new - names_old)
(OUT / "tactic_alias.json").write_text(json.dumps(
    {"alias": TACTIC_ALIAS, "old_only_labels": OLD_ONLY_LABELS, "new_only_labels": NEW_ONLY_LABELS}, indent=2))
print("Nhãn chỉ có ở bản cũ:", OLD_ONLY_LABELS, "| chỉ có ở bản mới:", NEW_ONLY_LABELS)

Nhãn chỉ có ở bản cũ: ['Defense Evasion'] | chỉ có ở bản mới: ['Defense Impairment', 'Stealth']


### 9.1. Mở rộng mỗi mục thành các truy vấn (mục × framing)

Mỗi truy vấn là đơn vị mà notebook 04 gửi cho mô hình. Ví dụ `D1-T1222|current` có `gold` = đáp án v19.2 và `other_gold` = đáp án v18.1.

In [17]:
def expand(it):
    out = []
    for fr, text in it["questions"].items():
        cur = fr in ("unversioned", "current")
        out.append(dict(query_id=f"{it['item_id']}|{fr}", item_id=it["item_id"], template=it["template"],
                        answer_type=it["answer_type"], framing=fr, direction="current" if cur else "historical",
                        target_version=V_NEW if cur else V_OLD, question=text,
                        gold=it["gold_new"] if cur else it["gold_old"],
                        gold_display=it["gold_display_new"] if cur else it["gold_display_old"],
                        other_gold=(it["gold_old"] if cur else it["gold_new"]) if it["anchor_old"] else it["gold_new"],
                        other_only_labels=(OLD_ONLY_LABELS if cur else NEW_ONLY_LABELS) if it["answer_type"] == "tactic_set" else [],
                        entity_names=it["entity_names"], pair_class=it["pair_class"],
                        change_types=it["change_types"], answer_changed=it["answer_changed"],
                        split=it["split"], lineage=it["lineage"], anchor_old=it["anchor_old"], evidence=it["evidence"]))
        # Có thể trả lời "theo phiên bản kia" không (STALE / ANACHRONISM)? Dùng để lọc khi tính tỉ lệ lỗi trôi
        out[-1]["drift_possible"] = bool(set(out[-1]["other_gold"]) - set(out[-1]["gold"]))
    return out

QUERIES_ALL = [q for it in ITEMS for q in expand(it)]
QBYID = {q["query_id"]: q for q in QUERIES_ALL}
print(f"{len(ITEMS)} mục → {len(QUERIES_ALL)} truy vấn")

2415 mục → 7190 truy vấn


### 9.2. Kiểm thử module chấm điểm trên các ví dụ thật

Các ví dụ dưới đây là chuyển đổi v18.1 → v19.2. Nếu em đổi `V_OLD`/`V_NEW`, các kiểm thử cụ thể này được bỏ qua; chỉ còn kiểm thử chung.

In [18]:
TA = TACTIC_ALIAS
# ---- Kiểm thử chung (không phụ thuộc phiên bản) ----------------------------------------------
for q in QUERIES_ALL:                                      # mục STABLE không thể bị chấm STALE
    if q["pair_class"] == "STABLE":
        assert set(q["gold"]) == set(q["other_gold"]) and not q["drift_possible"]
assert final_part("I think T1562.004 was replaced.\nANSWER: T1686") == ("T1686", True)

def fake(gold, other, atype="mitigation_set", direction="current", names=(), labels=()):
    return dict(gold=gold, other_gold=other, answer_type=atype, direction=direction,
                entity_names=list(names), other_only_labels=list(labels))
qm = fake(["M1038", "M1042"], ["M1038"])
assert score_drift(qm, "answer: m1038, m1042")["outcome"] == "CORRECT"                     # chữ thường
assert score_drift(qm, "**Answer:**\n- M1038 Execution Prevention\n- M1042 Disable or Remove Feature")["outcome"] == "CORRECT"
assert score_drift(qm, "ANSWER:\n\nM1038, M1042")["outcome"] == "CORRECT"                    # mã ở đoạn dưới
assert score_drift(qm, "ANSWER: M1038,\nM1042")["outcome"] == "CORRECT"                      # xuống dòng giữa danh sách
assert score_drift(qm, "ANSWER: M1038, M1042\n\nNote: M1050 was removed in later versions.")["outcome"] == "CORRECT"
assert score_drift(qm, "ANSWER: M1038, M1042\nANSWER: done")["outcome"] == "CORRECT"         # nhiều dòng ANSWER
assert score_drift(qm, "")["outcome"] == "NO_ANSWER"
qt = fake(["TA0011"], ["TA0011"], "tactic_set")
assert score_drift(qt, "ANSWER: Command and Control (TA0011), since it enables remote execution", TA)["outcome"] == "CORRECT"
assert score_drift(qt, "ANSWER: Command & Control", TA)["outcome"] == "CORRECT"
assert score_drift(qt, "ANSWER: COMMAND AND CONTROL", TA)["outcome"] == "CORRECT"
assert score_drift(qt, "ANSWER: Command and Control, used for pre-execution staging", TA)["outcome"] == "CORRECT"
qh = fake(["TA0005"], ["TA0112"], "tactic_set", "historical", labels=NEW_ONLY_LABELS)
r = score_drift(qh, "It belongs to Defense Evasion; attackers use it for stealth.", TA)       # văn xuôi, không có ANSWER
assert r["outcome"] == "CORRECT" and not r["label_drift"]                                  # "stealth" thường ≠ tên tactic
assert score_drift(qh, "ANSWER: defense evasion", TA)["outcome"] == "CORRECT"                 # chữ thường trong dòng ANSWER
assert score_drift(qt, "ANSWER: command and control", TA)["outcome"] == "CORRECT"
# Dòng giải thích sau một dòng ANSWER hoàn chỉnh không được tính
qd = fake(["T1686"], ["T1562.004"], "technique_set")
assert score_drift(qd, "ANSWER: T1686\n(was T1562.004)")["outcome"] == "CORRECT"
assert score_drift(qd, "ANSWER: T1686\nT1562.004 is revoked.")["outcome"] == "CORRECT"
assert score_drift(qd, "ANSWER: T1686 (formerly T1562.004)")["outcome"] == "STALE"            # quy ước: dòng ANSWER chỉ chứa đáp án
assert score_drift(qm, "ANSWER: M1038, M1042\n- Note: M1047 Audit was removed")["outcome"] == "CORRECT"
assert score_drift(qm, "**Answer:** The following mitigations apply:\n\n- M1038\n- M1042")["outcome"] == "CORRECT"
assert score_drift(qm, "ANSWER: M1038 Execution Prevention and\nM1042 Disable or Remove Feature or Program")["outcome"] == "CORRECT"
# Chữ thường trong câu văn của dòng ANSWER không bị hiểu là tactic
qs = fake(["TA0005"], ["TA0005"], "tactic_set")
assert score_drift(qs, "ANSWER: Stealth — it hides the impact of the attack", TA)["outcome"] == "CORRECT"
assert score_drift(qt, "ANSWER: Command and Control; used after initial access", TA)["outcome"] == "CORRECT"
assert not score_drift(qh, "ANSWER: TA0005 (Defense Evasion) — adversaries use it for stealth", TA)["label_drift"]
# Dòng ANSWER kết thúc bằng ':' / 'and' không nuốt phần giải thích
assert score_drift(qd, "ANSWER: T1686, for the following reason:\nT1562.004 was split in v19.")["outcome"] == "CORRECT"
assert score_drift(qd, "ANSWER: T1686 — note:\nT1562.004 is the old ID.")["outcome"] == "CORRECT"
assert score_drift(fake(["TA0112"], ["TA0005"], "tactic_set"), "ANSWER: TA0112 (Defense Impairment), and\npreviously it was TA0005.", TA)["outcome"] == "CORRECT"
# Danh sách mỗi dòng một mã
qm3 = fake(["M1022", "M1029", "M1041"], ["M1022"])
assert score_drift(qm3, "ANSWER: M1022\nM1029\nM1041")["outcome"] == "CORRECT"
assert score_drift(qm3, "ANSWER: M1022 Restrict File and Directory Permissions\nM1029 Remote Data Storage\nM1041 Encrypt Sensitive Information")["outcome"] == "CORRECT"
assert score_drift(qm3, "ANSWER: M1022\n  * M1029 (Remote Data Storage)\n  * M1041")["outcome"] == "CORRECT"
assert score_drift(fake(["TA0002", "TA0003"], ["TA0002"], "tactic_set"), "**Answer:**\n- Execution\n- Persistence", TA)["outcome"] == "CORRECT"
# Định dạng chính: JSON (guided decoding ở notebook 04)
assert score_drift(qd, '{"answer": ["T1686"]}')["outcome"] == "CORRECT"
assert score_drift(qd, '{"answer": ["T1686"]}')["format"] == "json"
assert score_drift(qd, 'Reasoning: T1562.004 was revoked.\n{"answer": ["t1686"], "note": "was T1562.004"}')["outcome"] == "CORRECT"
assert score_drift(qd, '{"answer": []}')["outcome"] == "NO_ANSWER"
assert score_drift(fake(["TA0112"], ["TA0005"], "tactic_set"), '{"answer": ["defense impairment"]}', TA)["outcome"] == "CORRECT"
assert score_drift(qd, '{"answer": ["T1686"], "meta": {"old": "T1562.004"}}')["outcome"] == "CORRECT"      # JSON lồng nhau
assert score_drift(qd, '{"reasoning": "the {old} id T1562.004 was revoked", "answer": ["T1686"]}')["outcome"] == "CORRECT"
assert score_drift(qd, '{"reasoning": "T1562.004 was revoked", "answer": "T1686"}')["outcome"] == "CORRECT"   # answer là chuỗi
assert score_drift(qd, '{"answer": null, "reasoning": "T1562.004 or T1686?"}')["outcome"] == "NO_ANSWER"
assert score_drift(qd, '```json\n{"answer": [{"id": "T1686", "replaces": "T1562.004"}]}\n```')["outcome"] == "CORRECT"
assert score_drift(qm, "ANSWER: M1038, M1042")["format"] == "marker"
# Dự phòng: các định dạng danh sách markdown thường gặp
assert score_drift(qm3, "ANSWER:\n- **M1022** Restrict File and Directory Permissions\n- **M1029** Remote Data Storage\n- **M1041** Encrypt Sensitive Information")["outcome"] == "CORRECT"
assert score_drift(qm3, "ANSWER:\n- Restrict File and Directory Permissions (M1022)\n- Remote Data Storage (M1029)\n- Encrypt Sensitive Information (M1041)")["outcome"] == "CORRECT"
assert score_drift(qm3, "ANSWER:\n| ID | Name |\n|---|---|\n| M1022 | Restrict |\n| M1029 | Remote |\n| M1041 | Encrypt |")["outcome"] == "CORRECT"
assert score_drift(qm3, "ANSWER: M1022\n- M1029: store logs remotely so they cannot be cleared\n- M1041")["outcome"] == "CORRECT"
# Dòng ANSWER cuối là từ chối trả lời -> không trả lời (không lùi về dòng trước)
assert score_drift(qd, "ANSWER: T1562.004 was revoked\nANSWER: I am not sure")["outcome"] == "NO_ANSWER"
# Che tên thực thể theo trọn từ: tên "At" không xoá chữ "at" bên trong "Privilege Escalation"
qa = fake(["TA0002", "TA0003", "TA0004"], ["TA0002", "TA0003", "TA0004"], "tactic_set", names=["At"])
assert score_drift(qa, "ANSWER: Execution, Persistence, Privilege Escalation", TA)["outcome"] == "CORRECT"
print("✓ Kiểm thử chung của module chấm điểm đều qua")

if (V_OLD, V_NEW) == ("18.1", "19.2"):
    q = QBYID["D1-T1222|current"]                         # TA0005 -> TA0112
    assert score_drift(q, "ANSWER: Defense Evasion", TA)["outcome"] == "STALE"
    assert score_drift(q, "ANSWER: Defense Evasion", TA)["label_drift"]
    assert score_drift(q, "Answer: Defense Impairment (TA0112)", TA)["outcome"] == "CORRECT"
    q = QBYID["D1-T1222|historical"]
    assert score_drift(q, "ANSWER: Defense Evasion (TA0005)", TA)["outcome"] == "CORRECT"
    assert score_drift(q, "ANSWER: Defense Impairment", TA)["outcome"] == "ANACHRONISM"
    q = QBYID["D1-T1027|current"]                         # cùng mã TA0005, đổi tên
    assert QBYID["D1-T1027|current"]["pair_class"] == "LABEL_CHANGED"
    r1 = score_drift(q, "ANSWER: Defense Evasion (TA0005)", TA); r2 = score_drift(q, "ANSWER: Stealth (TA0005)", TA)
    assert r1["outcome"] == r2["outcome"] == "CORRECT" and r1["label_drift"] and not r2["label_drift"]
    q = QBYID["D4-T1562.004|current"]                     # mã bị thu hồi
    assert score_drift(q, "ANSWER: T1686", TA)["outcome"] == "CORRECT"
    assert score_drift(q, "ANSWER: T1562.004", TA)["outcome"] == "STALE"
    assert score_drift(q, "T1562.004 was revoked.\nANSWER: T1686", TA)["outcome"] == "CORRECT"
    assert score_drift(QBYID["D4-T1562.004|historical"], "ANSWER: T1562.004", TA)["outcome"] == "CORRECT"
    q = QBYID["D2-T1070.001|current"]                     # đổi cha qua thu hồi: T1070 -> T1685
    assert score_drift(q, "ANSWER: T1070 Indicator Removal", TA)["outcome"] == "STALE"
    assert score_drift(q, "ANSWER: T1685", TA)["outcome"] == "CORRECT"
    # Tên thực thể được hỏi bị che: không bị hiểu nhầm thành tên tactic
    q = QBYID["D1-T1211|current"]                         # Exploitation for Defense Evasion -> ... for Stealth
    r = score_drift(q, "Exploitation for Defense Evasion (T1211).\nANSWER: Stealth (TA0005)", TA)
    assert r["outcome"] == "CORRECT" and not r["label_drift"]
    q = QBYID["D1-T1127|historical"]                      # Trusted Developer Utilities Proxy Execution
    assert score_drift(q, "ANSWER: Trusted Developer Utilities Proxy Execution belongs to Defense Evasion", TA)["outcome"] == "CORRECT"
    q = QBYID["D1-T1202|current"]                         # Indirect Command Execution: chỉ TA0005
    assert score_drift(q, "ANSWER: Indirect Command Execution is a Stealth technique", TA)["outcome"] == "CORRECT"
    # Kỹ thuật cấp cha trở thành sub-technique: có mục D2 NEW_ONLY
    assert QBYID["D2-T1684.001|current"]["gold"] == ["T1684"]
    # Mục mã cũ và mục mã kế nhiệm cùng tập dev/test
    assert QBYID["D1-T1562.002|current"]["split"] == QBYID["D1-T1685.001|current"]["split"]
    print("✓ Kiểm thử chấm điểm trên ví dụ v18.1 → v19.2 đều qua")
else:
    print("(bỏ qua kiểm thử riêng cho v18.1 → v19.2)")

✓ Kiểm thử chung của module chấm điểm đều qua
✓ Kiểm thử chấm điểm trên ví dụ v18.1 → v19.2 đều qua


## 10. Lấy mẫu cho pilot (notebook 04)

Pilot cần **vừa nhỏ vừa đủ mọi kiểu thay đổi**. Với mỗi mẫu câu hỏi:

- lấy **mọi** mục thuộc lớp thay đổi, tối đa `MAX_PER_CLASS` mục mỗi lớp;
- lấy thêm `CONTROLS_PER_TEMPLATE` mục `STABLE` làm **đối chứng**.

Mọi phép chọn dùng bộ sinh ngẫu nhiên riêng cho từng (mẫu, lớp), nên đổi tham số của một lớp không làm thay đổi lớp khác. Pilot dùng **cả dev và test** vì mục đích là quyết định đi tiếp hay dừng (go/no-go), chưa phải kết quả cuối. Trường `split` vẫn được giữ để thí nghiệm chính chỉ chỉnh tham số trên dev.

In [19]:
pilot_ids = []
for tpl in Q:
    for cls in CLASSES:
        pool = sorted(it["item_id"] for it in ITEMS if it["template"] == tpl and it["pair_class"] == cls)
        k = CONTROLS_PER_TEMPLATE if cls == "STABLE" else MAX_PER_CLASS
        rng = random.Random(f"{SEED}:drift-pilot:{tpl}:{cls}")
        pilot_ids += pool if len(pool) <= k else sorted(rng.sample(pool, k))
pilot_set = set(pilot_ids)
PILOT_ITEMS = [it for it in ITEMS if it["item_id"] in pilot_set]
PILOT_QUERIES = [q for it in PILOT_ITEMS for q in expand(it)]
pt = pd.DataFrame(PILOT_ITEMS).pivot_table(index="template", columns="pair_class", values="item_id", aggfunc="count", fill_value=0)
pt = pt.reindex(columns=[c for c in CLASSES if c in pt.columns]); pt["TỔNG"] = pt.sum(axis=1); pt.loc["TỔNG"] = pt.sum()
print(f"Pilot: {len(PILOT_ITEMS)} mục → {len(PILOT_QUERIES)} truy vấn "
      f"(framing: {dict(collections.Counter(q['framing'] for q in PILOT_QUERIES))})")
pt

Pilot: 435 mục → 1250 truy vấn (framing: {'unversioned': 435, 'current': 435, 'historical': 380})


pair_class,STABLE,ANSWER_CHANGED,LABEL_CHANGED,ANCHOR_RENAMED,REVOKED_ANCHOR,NEW_ONLY,TỔNG
template,,,,,,,
D1_tech_tactic,40,72,80,1,17,23,233
D2_subtech_parent,40,0,0,3,8,12,63
D3_tech_mitigation,40,0,0,4,14,20,78
D4_id_currency,40,0,0,4,17,0,61
TỔNG,160,72,80,12,56,55,435


## 11. Kho RAG trộn phiên bản (cho điều kiện "RAG trộn" ở notebook 04)

Thực tế, nhiều hệ thống **nạp thêm** bản ATT&CK mới vào chỉ mục cũ mà không xoá bản cũ. Kho trộn mô phỏng đúng tình huống đó:

- đoạn **giống hệt nhau** ở hai phiên bản (cùng mã, cùng nội dung, ví dụ mọi CVE và CWE) chỉ giữ **một bản**, mã `*|<mã đoạn>`;
- đoạn **khác nhau** được giữ **cả hai**, mã `<phiên bản>|<mã đoạn>`, ví dụ `18.1|T1222#0` và `19.2|T1222#0`;
- nội dung văn bản **không gắn thêm phiên bản**, giống một chỉ mục trộn ngây thơ. Phiên bản chỉ nằm ở `metadata`, để phương pháp ở Trụ 3 có thể dùng.

Hàm `mixed_id(v, cid)` đổi mã đoạn của một phiên bản sang mã trong kho trộn.

In [20]:
MIXED, SHARED = [], set()
for cid in sorted(set(CORPUS[V_OLD]) | set(CORPUS[V_NEW])):
    a, b = CORPUS[V_OLD].get(cid), CORPUS[V_NEW].get(cid)
    if a and b and a["text"] == b["text"] and a["title"] == b["title"]:
        SHARED.add(cid)
        MIXED.append({"_id": f"*|{cid}", "title": b["title"], "text": b["text"],
                      "metadata": {**b["metadata"], "versions": [V_OLD, V_NEW], "chunk_id": cid}})
    else:
        for v, c in ((V_OLD, a), (V_NEW, b)):
            if c:
                MIXED.append({"_id": f"{v}|{cid}", "title": c["title"], "text": c["text"],
                              "metadata": {**c["metadata"], "versions": [v], "chunk_id": cid}})

def mixed_id(v, cid):
    return f"*|{cid}" if cid in SHARED else f"{v}|{cid}"

kinds = collections.Counter((m["metadata"]["kind"], "chung" if len(m["metadata"]["versions"]) == 2 else "riêng") for m in MIXED)
print(f"Kho trộn: {len(MIXED)} đoạn ({len(SHARED)} đoạn chung, {len(MIXED) - len(SHARED)} đoạn riêng theo phiên bản)")
pd.Series(kinds).unstack(fill_value=0)

Kho trộn: 5206 đoạn (4239 đoạn chung, 967 đoạn riêng theo phiên bản)


,chung,riêng
capec,551,10
cve,1754,0
cwe,944,0
group,296,278
mitigation,174,108
technique,520,571


## 12. Ghi kết quả

| File | Nội dung |
|---|---|
| `items.jsonl` | toàn bộ quần thể mục ghép cặp |
| `queries_all.jsonl` | mọi truy vấn (mục × framing) |
| `pilot_items.jsonl`, `pilot_queries.jsonl` | mẫu cho pilot ở notebook 04 |
| `corpus_mixed.jsonl` | kho RAG trộn phiên bản |
| `population_by_class.csv` | bảng số mục theo mẫu × lớp (cho bài báo) |
| `tactic_alias.json` | bảng tên tactic hai phiên bản, dùng khi chấm |
| `manifest.json`, `DATASHEET.md` | nguồn, cấu hình, mã băm, kết quả kiểm tra |

In [21]:
def write_jsonl(p, rows):
    with open(p, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, sort_keys=True) + "\n")

for it in ITEMS:   # thêm mã đoạn trong kho trộn
    it["evidence_mixed"] = {v: [mixed_id(v, c) for c in it["evidence"][v]] for v in (V_OLD, V_NEW)}
for q in QUERIES_ALL + PILOT_QUERIES:
    q["evidence_mixed"] = {v: [mixed_id(v, c) for c in q["evidence"][v]] for v in (V_OLD, V_NEW)}

files = {"items.jsonl": ITEMS, "queries_all.jsonl": QUERIES_ALL, "pilot_items.jsonl": PILOT_ITEMS,
         "pilot_queries.jsonl": PILOT_QUERIES, "corpus_mixed.jsonl": MIXED}
for name, rows in files.items():
    write_jsonl(OUT / name, rows)
MANIFEST["counts"] = {"items": len(ITEMS), "queries_all": len(QUERIES_ALL), "pilot_items": len(PILOT_ITEMS),
                      "pilot_queries": len(PILOT_QUERIES), "corpus_mixed": len(MIXED), "corpus_shared": len(SHARED),
                      "excluded": {f"{a}:{b}": n for (a, b), n in sorted(EXCLUDED.items())},
                      "by_class": {f"{a}:{b}": int(n) for (a, b), n in df.groupby(["template", "pair_class"]).size().items()}}
MANIFEST["outputs"] = {p.name: sha256(p) for p in sorted(OUT.iterdir())
                       if p.suffix in (".jsonl", ".csv", ".json") and p.name not in ("manifest.json", "capec_attack_tag_status.csv")}
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))
for k, v in MANIFEST["outputs"].items():
    print(f"{v[:12]}…  {k}")

301ddae87bf7…  corpus_mixed.jsonl
b728835cf4e9…  items.jsonl
d76133e1cd33…  pilot_items.jsonl
1382ad00e2b4…  pilot_queries.jsonl
b3bbc8cc40f8…  population_by_class.csv
a1275541391f…  queries_all.jsonl
a5efc7448153…  tactic_alias.json


In [22]:
try:
    pop_md = tab.to_markdown()
except ImportError:
    pop_md = "```\n" + tab.to_string() + "\n```"
DATASHEET = f'''# Datasheet — CTI version-drift pairs v{DRIFTSET_VERSION} (ATT&CK v{V_OLD} → v{V_NEW})

## Motivation
Measure whether LLM/RAG CTI assistants answer according to the outdated or the current ATT&CK release when
the underlying knowledge changed between releases, broken down by change type.

## Composition
- {len(ITEMS)} paired items over 4 templates (technique→tactic, sub-technique→parent by name,
  technique→mitigations, technique-ID currency), each with gold answers for v{V_OLD} and v{V_NEW}.
- {len(QUERIES_ALL)} queries (item × framing: unversioned / current / historical).
- Pilot subset: {len(PILOT_ITEMS)} items / {len(PILOT_QUERIES)} queries (all change classes, capped at {MAX_PER_CLASS},
  plus {CONTROLS_PER_TEMPLATE} STABLE controls per template).
- Mixed-version RAG corpus: {len(MIXED)} chunks ({len(SHARED)} shared, the rest version-specific).

{pop_md}

## Collection process
Gold answers derived deterministically from MITRE ATT&CK STIX 2.1 (attack-stix-data) for both releases.
Revoked IDs resolved through official `revoked-by` chains, cross-checked against the revocation table of the
ATT&CK v19 release notes (17/17 identical, no one-to-many replacements). An optional manual table for
one-to-many splits exists for future releases ({len(SPLITS)} entries used here).
Change-type labels come from the release-diff ground truth (notebook 01). No LLM was used to write questions or answers.

## Preprocessing / exclusions
{json.dumps({f"{a}:{b}": n for (a, b), n in sorted(EXCLUDED.items())}, indent=1)}

## Uses
Measuring stale/anachronistic answers and stale labels by change type; evaluating version-aware retrieval.
Not intended as a general ATT&CK knowledge benchmark.

## Known limitations
- Gold follows MITRE's official one-to-one/many-to-one replacements. Scope dispersion (e.g. Impair Defenses'
  former sub-techniques now spread over T1685/T1686/T1688/T1689/T1690) is not encoded as a split.
- Revoked IDs never appear in the v{V_NEW} corpus ({MANIFEST["revoked_ids_mentioned_in_new_corpus"]} mentions), so
  current-only RAG cannot link old IDs to replacements lexically; this is a property being measured, not a defect.
- Questions cite entities as named in v{V_OLD} (analyst holding an older document); NEW_ONLY items use v{V_NEW} names.

## Maintenance
Regenerate with notebook 03 for any (old, new) release pair; manifest records SHA-256 of all inputs and outputs.
'''
(OUT / "DATASHEET.md").write_text(DATASHEET, encoding="utf-8")
MANIFEST["outputs"]["DATASHEET.md"] = sha256(OUT / "DATASHEET.md")
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))
print(DATASHEET[:1500])

# Datasheet — CTI version-drift pairs v1.2 (ATT&CK v18.1 → v19.2)

## Motivation
Measure whether LLM/RAG CTI assistants answer according to the outdated or the current ATT&CK release when
the underlying knowledge changed between releases, broken down by change type.

## Composition
- 2415 paired items over 4 templates (technique→tactic, sub-technique→parent by name,
  technique→mitigations, technique-ID currency), each with gold answers for v18.1 and v19.2.
- 7190 queries (item × framing: unversioned / current / historical).
- Pilot subset: 435 items / 1250 queries (all change classes, capped at 80,
  plus 40 STABLE controls per template).
- Mixed-version RAG corpus: 5206 chunks (4239 shared, the rest version-specific).

```
pair_class          STABLE  ANSWER_CHANGED  LABEL_CHANGED  ANCHOR_RENAMED  REVOKED_ANCHOR  NEW_ONLY  TỔNG
template                                                                                                 
D1_tech_tactic         475              72           

## 13. Phụ lục — thẻ ATT&CK lỗi thời trong CAPEC (xem trước cho Trụ 1)

CAPEC (MITRE) gắn mỗi mẫu tấn công với các mã ATT&CK. File STIX của CAPEC trong `mitre/cti` **không cập nhật cùng nhịp** với ATT&CK. Ô dưới đếm xem bao nhiêu mã ATT&CK mà CAPEC tham chiếu **đã bị thu hồi hoặc ngừng dùng** ở v18.1 và v19.2.

Trạng thái `unknown` nghĩa là mã không có trong ATT&CK **Enterprise**. Ví dụ T1513 thuộc ATT&CK Mobile, nên đây không phải thẻ lỗi thời.

Đây là **một ví dụ nhỏ của phép đo Trụ 1**: dữ liệu bên thứ ba vẫn mang mã cũ sau khi MITRE đã đổi. Ở Trụ 1 ta sẽ làm phép đo này trên quy mô lớn (luật Sigma, luật Elastic, báo cáo CTI…).

In [23]:
p = OPTIONAL["capec"]
if p.exists():
    def current_of(t):                     # mã hiện hành ở V_NEW của một mã bất kỳ (kể cả mã đã thu hồi từ trước)
        st = NEW["tech"].get(t, {}).get("status")
        if st == "active":
            return [t]
        if st == "revoked":
            tgt, path = resolve(NEW, t)
            for x in path:
                if x in SPLITS:
                    return SPLITS[x]
            return [tgt] if tgt in ACT_NEW else []
        return []
    cobjs = json.loads(p.read_text())["objects"]
    capec_mod = max((o.get("modified", "") for o in cobjs if o["type"] == "attack-pattern"), default="")
    rows = []
    for o in cobjs:
        if o["type"] != "attack-pattern" or o.get("revoked") or o.get("x_capec_status") == "Deprecated":
            continue
        cid = next((r["external_id"] for r in o.get("external_references", []) if r.get("source_name") == "capec"), None)
        for r in o.get("external_references", []):
            if r.get("source_name") == "ATTACK" and r.get("external_id"):
                t = r["external_id"] if r["external_id"].startswith("T") else "T" + r["external_id"]
                rows.append(dict(capec=cid, attack_id=t,
                                 **{f"status_{v}": S["tech"].get(t, {}).get("status", "unknown") for v, S in ((V_OLD, OLD), (V_NEW, NEW))},
                                 current=",".join(current_of(t))))
    cap = pd.DataFrame(rows)
    cap.to_csv(OUT / "capec_attack_tag_status.csv", index=False)
    MANIFEST["outputs"]["capec_attack_tag_status.csv"] = sha256(OUT / "capec_attack_tag_status.csv")
    MANIFEST["capec_last_modified"] = capec_mod[:10]
    (OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))
    print(f"CAPEC sửa đổi lần cuối: {capec_mod[:10]} | {cap.capec.nunique()} mẫu CAPEC, {len(cap)} liên kết, {cap.attack_id.nunique()} mã ATT&CK")
    display(pd.concat({f"v{V_OLD}": cap[f"status_{V_OLD}"].value_counts(), f"v{V_NEW}": cap[f"status_{V_NEW}"].value_counts()}, axis=1).fillna(0).astype(int))
    display(cap[cap[f"status_{V_NEW}"] != "active"].drop_duplicates("attack_id").sort_values("attack_id").head(20))
else:
    print("Không có data/raw/stix-capec.json — bỏ qua phụ lục")

CAPEC sửa đổi lần cuối: 2023-01-24 | 177 mẫu CAPEC, 272 liên kết, 189 mã ATT&CK


,v18.1,v19.2
active,270,256
revoked,1,15
unknown,1,1


,capec,attack_id,status_18.1,status_19.2,current
231,CAPEC-648,T1513,unknown,unknown,
181,CAPEC-578,T1562.001,active,revoked,T1685
65,CAPEC-268,T1562.002,active,revoked,T1685.001
9,CAPEC-13,T1562.003,active,revoked,T1690
183,CAPEC-578,T1562.004,active,revoked,T1686
171,CAPEC-571,T1562.006,active,revoked,T1685
184,CAPEC-578,T1562.007,active,revoked,T1686.001
67,CAPEC-268,T1562.008,active,revoked,T1685.002
186,CAPEC-578,T1562.009,active,revoked,T1688
219,CAPEC-641,T1574.002,revoked,revoked,T1574.001


## 14. Tóm tắt và việc tiếp theo

**Những gì notebook này tạo ra:**

1. Quần thể **mục ghép cặp** v18.1 ↔ v19.2 cho 4 mẫu câu hỏi, mỗi mục có đáp án ở hai phiên bản, lớp cặp và nhãn kiểu thay đổi khớp notebook 01.
2. **Ba cách hỏi** cho mỗi mục để đo tác động của việc nêu rõ phiên bản.
3. Module **`ctidrift_scoring.py`** với các nhãn kết quả `CORRECT / STALE / ANACHRONISM / PARTIAL / WRONG / NO_ANSWER` và cờ `label_drift`.
4. **Mẫu pilot** phân tầng theo lớp thay đổi, cùng **kho RAG trộn phiên bản**.
5. Một phát hiện ban đầu cho bài báo: kho bản mới **không còn dấu vết** của mã bị thu hồi.

**Việc tiếp theo:**

- **Em:** chạy notebook này, gửi thầy kết quả. Không cần lập `splits_manual.csv` cho v18.1 → v19.2.
- **Notebook 04 (pilot, quyết định đi tiếp hay dừng):** 2 mô hình mở chạy trên RTX 4070 × 3 điều kiện (không RAG / RAG một phiên bản / RAG trộn) × 3 cách hỏi, chấm bằng `ctidrift_scoring`. Tiêu chí *go*: tỉ lệ `STALE` ở các lớp thay đổi **cao rõ rệt** so với mức lỗi nền của nhóm `STABLE`, và khác nhau theo kiểu thay đổi.

**Gửi thầy:** notebook đã chạy (`03_cap_cau_hoi_truoc_sau.ipynb`) và file `benchmark/drift-18.1-19.2/manifest.json`.